# Logistic Regression - Script Ordered Storyboard

This notebook mirrors the `script.md` narration for Chapter 1 (The Sigmoid).

Each scene exports either a PNG or GIF into `renders/` so shots can be sequenced directly in voice-over order.

- We intentionally skip the truck/car recap visuals.
- The required students `(3h study, 2h exam, passed)` and `(2h study, 3h exam, failed)` are included.
- Parallel lines at `ST - EL = 1, 2, 3` (and negatives) tie to the narrated `60%`, `80%`, `100%` pass proportions.


In [146]:
import io
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from mpl_toolkits.mplot3d import Axes3D
from PIL import Image, ImageDraw

np.random.seed(7)

OUTPUT_DIR = Path("renders")
OUTPUT_DIR.mkdir(exist_ok=True)

# Typography / padding (single source of truth for exports)
FONT_SIZE = 11
AXIS_LABEL_SIZE = 12
LEGEND_SIZE = 10
TITLE_SIZE = 12
NOTE_SIZE = 10
ANNOT_SIZE = 9
SAVE_PAD_INCHES = 0.12

# Fixed axes box for exponential GIFs (25/26): avoids bbox_tight crop shift when tick labels widen.
EXP_FIGSIZE = (8, 5.2)
EXP_SUBPLOT_ADJ = dict(left=0.09, right=0.91, bottom=0.09, top=0.91)


def finalize_exponential_figure(fig):
    fig.subplots_adjust(**EXP_SUBPLOT_ADJ)


plt.rcParams.update(
    {
        "font.size": FONT_SIZE,
        "axes.labelsize": AXIS_LABEL_SIZE,
        "axes.titlesize": TITLE_SIZE,
        "axes.labelpad": 8.0,
        "axes.titlepad": 10.0,
        "legend.fontsize": LEGEND_SIZE,
        "xtick.labelsize": FONT_SIZE,
        "ytick.labelsize": FONT_SIZE,
        "legend.frameon": True,
        "legend.framealpha": 0.96,
        "legend.borderaxespad": 0.55,
        "legend.labelspacing": 0.35,
        "legend.handlelength": 1.35,
        "legend.handletextpad": 0.65,
        "savefig.pad_inches": SAVE_PAD_INCHES,
    }
)


PASS_COLOR = "#2ca02c"
FAIL_COLOR = "#d62728"
NEUTRAL_COLOR = "#4f4f4f"

CHECK_ICON_PATH = OUTPUT_DIR / "check.png"
CROSS_ICON_PATH = OUTPUT_DIR / "cross.png"


def _ensure_outcome_icons(size=96, line_width=12):
    if not CHECK_ICON_PATH.exists():
        img = Image.new("RGBA", (size, size), (255, 255, 255, 0))
        draw = ImageDraw.Draw(img)
        draw.line([(20, 55), (42, 76), (78, 24)], fill=(44, 160, 44, 255), width=line_width)
        img.save(CHECK_ICON_PATH)

    if not CROSS_ICON_PATH.exists():
        img = Image.new("RGBA", (size, size), (255, 255, 255, 0))
        draw = ImageDraw.Draw(img)
        draw.line([(24, 24), (72, 72)], fill=(214, 39, 40, 255), width=line_width)
        draw.line([(72, 24), (24, 72)], fill=(214, 39, 40, 255), width=line_width)
        img.save(CROSS_ICON_PATH)


_ensure_outcome_icons()
CHECK_ICON = np.asarray(Image.open(CHECK_ICON_PATH).convert("RGBA"))
CROSS_ICON = np.asarray(Image.open(CROSS_ICON_PATH).convert("RGBA"))


def add_outcome_icon(ax, x, y_value, passed, zoom=0.16, alpha=1.0):
    image_array = CHECK_ICON if passed else CROSS_ICON
    icon = OffsetImage(image_array, zoom=zoom)
    icon.set_alpha(alpha)
    ab = AnnotationBbox(icon, (x, y_value), frameon=False)
    ax.add_artist(ab)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def fig_to_image(fig, dpi=140, tight_layout=True):
    buf = io.BytesIO()
    kw = {"format": "png", "dpi": dpi, "pad_inches": SAVE_PAD_INCHES}
    if tight_layout:
        kw["bbox_inches"] = "tight"
    else:
        kw["bbox_inches"] = None
    fig.savefig(buf, **kw)
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def save_gif(images, filename, duration=40):
    if not images:
        raise ValueError("images list is empty")
    images[0].save(
        OUTPUT_DIR / filename,
        save_all=True,
        append_images=images[1:],
        duration=duration,
        loop=0,
    )


In [163]:
# Two-stage dataset design:
# 1) Before "that's not realistic": linearly separable points.
# 2) After that line: add symmetric noisy students to break separability.

separable_points = [
    # Fail class (left of boundary, z=study-exam < 0)
    (2, 3, 0),  # required fail example
    (4, 5, 0), (5, 6, 0),
    (1, 3, 0), (2, 4, 0), (4, 6, 0),
    (1, 4, 0), (3, 6, 0), (1, 6, 0),


    # Pass class (right of boundary, z > 0)
    (3, 2, 1),  # required pass example
    (5, 4, 1), (6, 5, 1),                    # z=+1 -> 3 pass
    (4, 2, 1), (6, 4, 1), (3, 1, 1),  # z=+2 -> 4 pass
    (4, 1, 1), (5, 2, 1), (6, 3, 1),  # z=+3 -> 4 pass
    (6, 2, 1),  # z=+4 (Scene 6b anchor)
    (6, 1, 1),
]

# Added noisy students (symmetric around z=0, on parallel lines)
# to break strict linear separability after "but that's not realistic".
noisy_symmetric_points = [
    # symmetric pairs on +/-1
    (2, 1, 0),  # z=+1 but fails
    (1, 2, 1),  # z=-1 but passes
    (3, 4, 1),  # z=-2 but fails
    (4, 3, 0),  # z=+2 but passes

    # symmetric pair on +/-2
    (3, 5, 1),  # z=+2 but passes
    (5, 3, 0),  # z=-2 but fails
]

realistic_points = separable_points + noisy_symmetric_points


def unpack_points(point_list):
    arr = np.array(point_list, dtype=float)
    s = arr[:, 0]
    e = arr[:, 1]
    labels = arr[:, 2].astype(int)
    diff = s - e
    return s, e, labels, diff


study_sep, exam_sep, y_sep, z_sep = unpack_points(separable_points)
study_real, exam_real, y_real, z_real = unpack_points(realistic_points)

xlim = (0, 7)
ylim = (0, 7)

midpoint_shift = (z_sep[y_sep == 0].max() + z_sep[y_sep == 1].min()) / 2.0

print(f"Separable students: {len(separable_points)}")
print(f"Realistic students: {len(realistic_points)}")
print("Required pass example present:", np.any((study_sep == 3) & (exam_sep == 2) & (y_sep == 1)))
print("Required fail example present:", np.any((study_sep == 2) & (exam_sep == 3) & (y_sep == 0)))
print("Threshold midpoint between classes (separable stage):", midpoint_shift)


Separable students: 20
Realistic students: 26
Required pass example present: True
Required fail example present: True
Threshold midpoint between classes (separable stage): 0.0


In [182]:
def draw_dataset(ax, study_vals, exam_vals, labels, mask=None, alpha=0.95, title=None):
    if mask is None:
        mask = np.ones(len(study_vals), dtype=bool)

    for i in np.where(mask)[0]:
        add_outcome_icon(ax, study_vals[i], exam_vals[i], passed=bool(labels[i]), alpha=alpha)

    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.grid(alpha=0.2)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)


def add_combined_legend(ax, loc="upper left"):
    line_handles, line_labels = ax.get_legend_handles_labels()
    merged_handles = []
    merged_labels = []
    seen = set()

    for h, lbl in zip(line_handles, line_labels):
        if lbl and lbl not in seen:
            merged_handles.append(h)
            merged_labels.append(lbl)
            seen.add(lbl)

    if merged_handles:
        ax.legend(
            handles=merged_handles,
            labels=merged_labels,
            loc=loc,
            prop={"size": LEGEND_SIZE},
        )


def add_threshold_line(ax, shift=0.0, label=None, style="--", color=NEUTRAL_COLOR, linewidth=1.0, x_range=None):
    x0, x1 = x_range if x_range is not None else xlim
    x = np.linspace(x0, x1, 200)
    y_line = x - shift
    ax.plot(x, y_line, style, color=color, linewidth=linewidth, label=label)



def draw_axes_only(ax):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.grid(alpha=0.2)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)


def draw_axes_only_study_bold(ax):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel(
        "Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=8, fontweight="bold"
    )
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.grid(alpha=0.2)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)


def draw_axes_only_exam_bold(ax):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.set_ylabel(
        "Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=8, fontweight="bold"
    )
    ax.grid(alpha=0.2)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)


def shade_pass_half(ax, shift=0.0, alpha=0.22):
    xa, xb = xlim
    ya, yb = ylim
    xs = np.linspace(xa, xb, 400)
    y_line = xs - shift
    top = np.minimum(y_line, yb)
    bot = np.full_like(xs, ya)
    ax.fill_between(xs, bot, top, where=(top > bot), alpha=alpha, color=PASS_COLOR, linewidth=0, zorder=0)


def shade_fail_half(ax, shift=0.0, alpha=0.22):
    xa, xb = xlim
    ya, yb = ylim
    xs = np.linspace(xa, xb, 400)
    y_line = xs - shift
    bot2 = np.maximum(y_line, ya)
    ax.fill_between(xs, bot2, yb, where=(yb > bot2), alpha=alpha, color=FAIL_COLOR, linewidth=0, zorder=0)


def draw_z_axis_panel(ax, z_value, label):
    ax.axhline(0, color="black", linewidth=1)
    ax.scatter([z_value], [0], color="#ff7f0e", s=120, zorder=5)
    ax.text(z_value + 0.05, 0.08, label, fontsize=NOTE_SIZE)
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-0.6, 0.6)
    ax.set_yticks([])
    ax.set_xlabel("ST - EL", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.tick_params(axis="x", which="major", labelsize=FONT_SIZE)
    ax.grid(alpha=0.15)


def save_fig(fig, filename):
    fig.savefig(
        OUTPUT_DIR / filename,
        dpi=160,
        bbox_inches="tight",
        pad_inches=SAVE_PAD_INCHES,
    )
    plt.close(fig)


## Scene 1 - Introduce the dataset

Script alignment: survey students, collect study time, exam length, and pass/fail outcome.

The **next code cell** exports `00`–`16` (axis-label emphasis, plain axes, staged collection order, boundary crossing, then threshold visuals). The two cells after that export the full overview plots (`17`–`18`).


In [149]:
# Intro montage (exports 00–16): label beats, axes-only, staged build, crossing story, threshold pedagogy.
rng_build = np.random.default_rng(7)
n_sep = len(study_sep)

AXIS_LABEL_HOLD = 22

# ---- 00: axes only — Study time label bold (no data) ----
frames_axis_labels = []
for _ in range(AXIS_LABEL_HOLD):
    fig, ax = plt.subplots(figsize=(8, 6))
    draw_axes_only_study_bold(ax)
    frames_axis_labels.append(fig_to_image(fig))
save_gif(frames_axis_labels, "00_axes_study_time_bold.gif", duration=35)

# ---- 01: axes only — Exam length label bold (no data) ----
frames_axis_labels = []
for _ in range(AXIS_LABEL_HOLD):
    fig, ax = plt.subplots(figsize=(8, 6))
    draw_axes_only_exam_bold(ax)
    frames_axis_labels.append(fig_to_image(fig))
save_gif(frames_axis_labels, "01_axes_exam_length_bold.gif", duration=35)

# ---- 02: axes only (no data), both labels regular weight ----
fig, ax = plt.subplots(figsize=(8, 6))
draw_axes_only(ax)
save_fig(fig, "02_axes_only.png")

# ---- 03: dataset build — (3,2,1) then (2,3,0) then random order ----
idx_pass_req = int(np.where((study_sep == 3) & (exam_sep == 2) & (y_sep == 1))[0][0])
idx_fail_req = int(np.where((study_sep == 2) & (exam_sep == 3) & (y_sep == 0))[0][0])
rest = [i for i in range(n_sep) if i not in (idx_pass_req, idx_fail_req)]
rng_build.shuffle(rest)
reveal_order = [idx_pass_req, idx_fail_req] + rest

frames = []
# Hold empty axes, then one new point per step; first two points linger much longer.
DATASET_EMPTY_HOLD = 24
DATASET_FIRST_TWO_HOLD = 48
DATASET_REST_HOLD = 2


def _append_dataset_build_frames(mask, n_repeat):
    for _ in range(n_repeat):
        fig, ax = plt.subplots(figsize=(8, 6))
        draw_dataset(ax, study_sep, exam_sep, y_sep, mask=mask)
        add_combined_legend(ax, loc="upper left")
        frames.append(fig_to_image(fig))


mask = np.zeros(n_sep, dtype=bool)
_append_dataset_build_frames(mask, DATASET_EMPTY_HOLD)

for k in range(1, n_sep + 1):
    mask = np.zeros(n_sep, dtype=bool)
    mask[reveal_order[:k]] = True
    hold = DATASET_FIRST_TWO_HOLD if k <= 2 else DATASET_REST_HOLD
    _append_dataset_build_frames(mask, hold)

save_gif(frames, "03_dataset_build.gif", duration=35)

# ---- 02: failed point pushed right (more study); threshold/boundary not drawn ----
EXAM_ANIM = 3.0
ST_START, ST_END = 2.0, 3.25
N_ANIM = 55
PUSH_HOLD = 5
st_values = np.concatenate([np.linspace(ST_START, ST_END, N_ANIM), np.full(PUSH_HOLD, ST_END)])
static_mask = np.ones(n_sep, dtype=bool)
static_mask[idx_fail_req] = False
frames = []
for st in st_values:
    passed = (st - EXAM_ANIM) > 1e-6
    fig, ax = plt.subplots(figsize=(8, 6))
    draw_dataset(ax, study_sep, exam_sep, y_sep, mask=static_mask, alpha=0.28)
    add_outcome_icon(ax, st, EXAM_ANIM, passed=passed, zoom=0.24, alpha=1.0)
    add_combined_legend(ax, loc="upper left")
    frames.append(fig_to_image(fig))
save_gif(frames, "04_fail_point_crosses_threshold.gif", duration=42)

# ---- 05: threshold, unlabeled ----
fig, ax = plt.subplots(figsize=(8, 6))
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label=None, color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "05_threshold_unlabeled.png")

# ---- 06: unlabeled threshold + pass half green ----
fig, ax = plt.subplots(figsize=(8, 6))
shade_pass_half(ax, shift=midpoint_shift)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label=None, color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "06_threshold_green_right.png")

# ---- 07: unlabeled threshold + fail half red ----
fig, ax = plt.subplots(figsize=(8, 6))
shade_fail_half(ax, shift=midpoint_shift)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label=None, color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "07_threshold_red_left.png")

# ---- 08: threshold drifts slightly (still unlabeled) ----
drift = np.linspace(-0.99, 0.99, 100)
frames = []
for sh in drift:
    fig, ax = plt.subplots(figsize=(8, 6))
    draw_dataset(ax, study_sep, exam_sep, y_sep)
    add_threshold_line(ax, shift=midpoint_shift + sh, label=None, color="black", linewidth=1)
    add_combined_legend(ax, loc="upper left")
    frames.append(fig_to_image(fig))
save_gif(frames, "08_threshold_micro_drift.gif", duration=40)

# ---- 09: add (3,3) pass; threshold shifted so classes still separated ----
SHIFT_FOR_33 = -0.28
st_33 = np.append(study_sep, 3.0)
ex_33 = np.append(exam_sep, 3.0)
y_33 = np.append(y_sep, 1)
fig, ax = plt.subplots(figsize=(8, 6))
draw_dataset(ax, st_33, ex_33, y_33)
add_threshold_line(ax, shift=midpoint_shift + SHIFT_FOR_33, label=None, color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "09_dataset_extra_pass_shifted_threshold.png")

# ---- 10: (3,3) pass + (4,3) fail, no threshold ----
st_3343 = np.append(st_33, 4.0)
ex_3343 = np.append(ex_33, 3.0)
y_3343 = np.append(y_33, 0)
fig, ax = plt.subplots(figsize=(8, 6))
draw_dataset(ax, st_3343, ex_3343, y_3343)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "10_dataset_two_extra_no_threshold.png")

# ---- 11: threshold labeled ST = EL ----
fig, ax = plt.subplots(figsize=(8, 6))
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST = EL", color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "11_threshold_labeled_st_eq_el.png")

# ---- 12: ST = EL + black circle at (3,3) ----
fig, ax = plt.subplots(figsize=(8, 6))
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST = EL", color="black", linewidth=1)
ax.plot([3], [3], marker="o", markersize=11, markerfacecolor="black", markeredgecolor="black", linestyle="none", zorder=6)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "12_threshold_st_eq_el_dot_33.png")

# ---- 13: labeled threshold + green pass half + ST > EL ----
fig, ax = plt.subplots(figsize=(8, 6))
shade_pass_half(ax, shift=midpoint_shift)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST = EL", color="black", linewidth=1)
ax.text(4.5, 3.0, "ST > EL", fontsize=AXIS_LABEL_SIZE, color="#145214", fontweight="bold")
add_combined_legend(ax, loc="upper left")
save_fig(fig, "13_threshold_green_labeled_st_gt_el.png")

# ---- 14: labeled threshold + red fail half + ST < EL ----
fig, ax = plt.subplots(figsize=(8, 6))
shade_fail_half(ax, shift=midpoint_shift)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST = EL", color="black", linewidth=1)
ax.text(2.0, 5.0, "ST < EL", fontsize=AXIS_LABEL_SIZE, color="#7a1212", fontweight="bold")
add_combined_legend(ax, loc="upper left")
save_fig(fig, "14_threshold_red_labeled_st_lt_el.png")

# ---- 15: line labeled ST - EL = EL - EL ----
fig, ax = plt.subplots(figsize=(8, 6))
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = EL - EL", color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "15_threshold_label_st_minus_el_eq_el_minus_el.png")

# ---- 16: line labeled ST - EL = 0 ----
fig, ax = plt.subplots(figsize=(8, 6))
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "16_threshold_label_st_minus_el_eq_0.png")


In [150]:
fig, ax = plt.subplots(figsize=(8, 6))
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label=f"ST = EL", color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "17_dataset_overview.png")


In [151]:
fig, ax = plt.subplots(figsize=(8, 6))
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label=f"ST - EL = {midpoint_shift:.1f}", color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "18_dataset_overview_labeled.png")


## Scene 4 - Parallel lines (separate additive plots)

Script alignment: **`19`** animates markers on the **ST - EL = 0** diagonal from **(1,1)** through **(6,6)**; **`22`** highlights **(2,1)** with the same black marker style, then bold ticks **2** (study time) and **1** (exam length); then each line for fixed `ST - EL` appears one-by-one in additive order (**0, 1, 2, 3, 4**; no **-1** line). **`25_fail_point_crosses_threshold_st_el_0.gif`** replays the intro fail-point push (**`04`**) with **ST - EL = 0** drawn.


In [155]:
# ---- 19: GIF — dataset + ST-EL=0 line; black markers appear on diagonal (1,1)…(6,6) one at a time ----
DIAG_PT_HOLD = 4
DIAG_BASE_HOLD = 3
DIAG_MS = 40
diag_pts = [(float(i), float(i)) for i in range(1, 7)]

frames19 = []

for _ in range(DIAG_BASE_HOLD):
    fig, ax = plt.subplots(figsize=(8, 6))
    draw_dataset(ax, study_sep, exam_sep, y_sep)
    add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
    add_combined_legend(ax, loc="upper left")
    frames19.append(fig_to_image(fig))

for k in range(1, len(diag_pts) + 1):
    for _ in range(DIAG_PT_HOLD):
        fig, ax = plt.subplots(figsize=(8, 6))
        draw_dataset(ax, study_sep, exam_sep, y_sep)
        add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
        for tx, ty in diag_pts[:k]:
            ax.plot(
                [tx],
                [ty],
                marker="o",
                markersize=11,
                markerfacecolor="black",
                markeredgecolor="black",
                linestyle="none",
                zorder=6,
            )
        add_combined_legend(ax, loc="upper left")
        frames19.append(fig_to_image(fig))

save_gif(frames19, "19_dataset_points_on_threshold.gif", duration=DIAG_MS)

# ---- 20–21: same half-plane emphasis as 13/14, labels ST - EL > 0 / ST - EL < 0 ----
fig, ax = plt.subplots(figsize=(8, 6))
shade_pass_half(ax, shift=midpoint_shift)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
ax.text(
    4.5,
    3.0,
    "ST - EL > 0",
    fontsize=AXIS_LABEL_SIZE,
    color="#145214",
    fontweight="bold",
)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "20_threshold_green_st_minus_el_gt_0.png")

fig, ax = plt.subplots(figsize=(8, 6))
shade_fail_half(ax, shift=midpoint_shift)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
ax.text(
    2.0,
    5.0,
    "ST - EL < 0",
    fontsize=AXIS_LABEL_SIZE,
    color="#7a1212",
    fontweight="bold",
)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "21_threshold_red_st_minus_el_lt_0.png")

# ---- 22: GIF — separable dataset + threshold; black marker at (2,1); bold ST tick 2 then EL tick 1 ----
HIGHLIGHT_MS = 40
HIGHLIGHT_HOLD = 5


def _bold_tick_at_value(ax, axis, value):
    labels = ax.get_xticklabels() if axis == "x" else ax.get_yticklabels()
    for lbl in labels:
        t = lbl.get_text().strip()
        try:
            if abs(float(t) - float(value)) < 1e-9:
                lbl.set_fontweight("bold")
        except ValueError:
            continue


def _frame_dataset_highlight(ax, *, show_marker, bold_st2, bold_el1):
    draw_dataset(ax, study_sep, exam_sep, y_sep)
    add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
    if show_marker:
        ax.plot(
            [2.0],
            [1.0],
            marker="o",
            markersize=11,
            markerfacecolor="black",
            markeredgecolor="black",
            linestyle="none",
            zorder=6,
        )
    add_combined_legend(ax, loc="upper left")
    if bold_st2:
        _bold_tick_at_value(ax, "x", 2)
    if bold_el1:
        _bold_tick_at_value(ax, "y", 1)


frames_highlight = []
for _ in range(HIGHLIGHT_HOLD):
    fig, ax = plt.subplots(figsize=(8, 6))
    _frame_dataset_highlight(ax, show_marker=False, bold_st2=False, bold_el1=False)
    frames_highlight.append(fig_to_image(fig))
for _ in range(HIGHLIGHT_HOLD):
    fig, ax = plt.subplots(figsize=(8, 6))
    _frame_dataset_highlight(ax, show_marker=True, bold_st2=False, bold_el1=False)
    frames_highlight.append(fig_to_image(fig))
for _ in range(HIGHLIGHT_HOLD):
    fig, ax = plt.subplots(figsize=(8, 6))
    _frame_dataset_highlight(ax, show_marker=True, bold_st2=True, bold_el1=False)
    frames_highlight.append(fig_to_image(fig))
for _ in range(HIGHLIGHT_HOLD):
    fig, ax = plt.subplots(figsize=(8, 6))
    _frame_dataset_highlight(ax, show_marker=True, bold_st2=True, bold_el1=True)
    frames_highlight.append(fig_to_image(fig))

save_gif(frames_highlight, "22_dataset_st2_el1_marker_ticks.gif", duration=HIGHLIGHT_MS)

line_specs = [
    (0, "black", "ST - EL = 0"),
    (1, "#1f77b4", "ST - EL = 1"),
    (2, "#17becf", "ST - EL = 2"),
    (3, "#2ca02c", "ST - EL = 3"),
    (4, "#bcbd22", "ST - EL = 4"),
]

PARALLEL_HOLD = 4

frames = []
for step in range(1, len(line_specs) + 1):
    fig, ax = plt.subplots(figsize=(8, 6))
    draw_dataset(ax, study_sep, exam_sep, y_sep)

    for shift, color, label in line_specs[:step]:
        add_threshold_line(ax, shift=shift, label=label, color=color, linewidth=1)

    add_combined_legend(ax, loc="upper left")
    file_name = f"23_parallel_lines_step_{step:02d}.png"
    save_fig(fig, file_name)

    frame = Image.open(OUTPUT_DIR / file_name).convert("RGB")
    for _ in range(PARALLEL_HOLD):
        frames.append(frame)

save_gif(frames, "24_parallel_lines_additive.gif", duration=40)

# ---- 25: like `04_fail_point_crosses_threshold.gif` but draw ST - EL = 0 (same motion at EL=3) ----
frames = []
for st in st_values:
    passed = (st - EXAM_ANIM) > 1e-6
    fig, ax = plt.subplots(figsize=(8, 6))
    draw_dataset(ax, study_sep, exam_sep, y_sep, mask=static_mask, alpha=0.28)
    add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
    add_outcome_icon(ax, st, EXAM_ANIM, passed=passed, zoom=0.24, alpha=1.0)
    add_combined_legend(ax, loc="upper left")
    frames.append(fig_to_image(fig))
save_gif(frames, "25_fail_point_crosses_threshold_st_el_0.gif", duration=42)


## Scene 5 - "But that's not realistic" (add symmetric noisy students)

Script alignment:
- realistic students break perfect separability while keeping the same midpoint threshold
- `26_not_realistic_transition.gif`: **separable** dataset + threshold first; only the **new** (noisy) students appear one at a time — first **(4,3)** then **(3,4)**, then the rest (axis ticks/labels + legend like other dataset plots)
- `27_sigmoid_colormap_topdown_reveal.gif`: top-down **σ(0.5·ST − 0.5·EL)** colormap (same style as the final static top-down PNG): smooth reveal of **ST − EL > 0**, then **ST − EL < 0**, with threshold and outcome icons
- `28_parallel_lines_additive_realistic.gif`: same additive cadence as **`24_parallel_lines_additive.gif`**, on the **full realistic** dataset; only **`ST - EL = 0`** is legend-labeled; colored guides through **`ST - EL = 3`** (no `z=4` line)
- `29_proportions_60_80_100.gif`: **full** realistic dataset + **black** threshold throughout; colored `z=1,2,3` lines and pass-rate text animate **on top** (same counting beats as before; same axes/legend as other dataset plots)
- narrated proportions for `ST - EL = 1, 2, 3` remain `60%`, `80%`, `100%`


In [156]:
# Scene 5 — "not realistic": separable dataset + threshold first; noisy points one-by-one: (4,3) then (3,4) then lexsorted rest; full axis labels/ticks + legend.
SC5_MS = 40
HOLD_BASELINE = 18
HOLD_PER_POINT = 2
n_r = len(study_real)
n_sep = len(study_sep)
new_idx = np.arange(n_sep, n_r)
priority_pairs = ((4.0, 3.0), (3.0, 4.0))
priority = []
for s0, e0 in priority_pairs:
    hit = np.where((study_real == s0) & (exam_real == e0))[0]
    if hit.size:
        priority.append(int(hit[0]))
rest = [int(i) for i in new_idx if int(i) not in priority]
rest_arr = np.array(rest, dtype=int)
if rest_arr.size:
    rest_sorted = rest_arr[np.lexsort((exam_real[rest_arr], study_real[rest_arr]))].tolist()
else:
    rest_sorted = []
reveal_new = np.array(priority + rest_sorted, dtype=int)

frames = []
for _ in range(HOLD_BASELINE):
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.grid(alpha=0.12)
    add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
    mask_sep = np.zeros(n_r, dtype=bool)
    mask_sep[:n_sep] = True
    draw_dataset(ax, study_real, exam_real, y_real, mask=mask_sep, alpha=0.95)
    add_combined_legend(ax, loc="upper left")
    frames.append(fig_to_image(fig))

for k in range(1, len(reveal_new) + 1):
    mask = np.zeros(n_r, dtype=bool)
    mask[:n_sep] = True
    mask[reveal_new[:k]] = True
    for _ in range(HOLD_PER_POINT):
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        ax.grid(alpha=0.12)
        add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
        draw_dataset(ax, study_real, exam_real, y_real, mask=mask, alpha=0.95)
        add_combined_legend(ax, loc="upper left")
        frames.append(fig_to_image(fig))

save_gif(frames, "26_not_realistic_transition.gif", duration=SC5_MS)

# ---- 27: top-down colormap σ(0.5·ST − 0.5·EL) — smooth reveal ST−EL > 0, then ST−EL < 0 (same layout as final top-down PNG) ----
import matplotlib as mpl

TOPDOWN_REVEAL_MS = 42
GRID_TD = 220
st_td = np.linspace(xlim[0], xlim[1], GRID_TD)
el_td = np.linspace(ylim[0], ylim[1], GRID_TD)
ST_td, EL_td = np.meshgrid(st_td, el_td)
DIFF_td = ST_td - EL_td
Z_td = sigmoid(0.5 * ST_td - 0.5 * EL_td)
cvals_td = [0, 0.5, 1]
colors_td = ["red", "white", "green"]
norm_td = plt.Normalize(min(cvals_td), max(cvals_td))
tuples_td = list(zip(map(norm_td, cvals_td), colors_td))
CMAP_td = mpl.colors.LinearSegmentedColormap.from_list("", tuples_td, 100)
dmax_td = float(np.max(DIFF_td))
dmin_td = float(np.min(DIFF_td))
eps_td = 0.05 * max(dmax_td, abs(dmin_td))


def _ease_smooth(t):
    t = np.clip(t, 0.0, 1.0)
    return 0.5 - 0.5 * np.cos(np.pi * t)


def _fig_topdown_masked(zm):
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.contourf(
        ST_td,
        EL_td,
        zm,
        levels=np.linspace(0.0, 1.0, 45),
        cmap=CMAP_td,
        vmin=0,
        vmax=1,
        antialiased=True,
        alpha=0.32,
    )
    add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
    for s, e, lbl in zip(study_real, exam_real, y_real):
        add_outcome_icon(ax, float(s), float(e), passed=bool(lbl), zoom=0.16, alpha=0.95)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.grid(alpha=0.12)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
    add_combined_legend(ax, loc="upper left")
    return fig_to_image(fig)


frames_td = []
N_PASS = 52
N_FAIL = 52
HOLD_PASS_FULL = 10
HOLD_ALL_FULL = 14

for j in range(N_PASS):
    u = eps_td + (dmax_td - eps_td) * _ease_smooth(j / max(N_PASS - 1, 1))
    zm = np.where((DIFF_td > 0) & (DIFF_td <= u), Z_td, np.nan)
    frames_td.append(_fig_topdown_masked(zm))

zm_pass_done = np.where(DIFF_td > 0, Z_td, np.nan)
for _ in range(HOLD_PASS_FULL):
    frames_td.append(_fig_topdown_masked(zm_pass_done))

for j in range(N_FAIL):
    lo = -eps_td + (dmin_td + eps_td) * _ease_smooth(j / max(N_FAIL - 1, 1))
    zm = np.where(
        DIFF_td > 0,
        Z_td,
        np.where((DIFF_td >= lo) & (DIFF_td < 0), Z_td, np.nan),
    )
    frames_td.append(_fig_topdown_masked(zm))

zm_all = Z_td
for _ in range(HOLD_ALL_FULL):
    frames_td.append(_fig_topdown_masked(zm_all))

save_gif(frames_td, "27_sigmoid_colormap_topdown_reveal.gif", duration=TOPDOWN_REVEAL_MS)

# ---- 28: like `24_parallel_lines_additive.gif` but full **realistic** dataset; only z=0 labeled; lines through z=3 (no z=4) ----
line_specs_realistic = [
    (0, "black", "ST - EL = 0"),
    (1, "#1f77b4", None),
    (2, "#17becf", None),
    (3, "#2ca02c", None),
]
PARALLEL_REAL_MS = 40
PARALLEL_REAL_HOLD = 4
frames_pr = []
for step in range(1, len(line_specs_realistic) + 1):
    fig, ax = plt.subplots(figsize=(8, 6))
    draw_dataset(ax, study_real, exam_real, y_real, mask=np.ones(n_r, dtype=bool), alpha=0.95)
    for shift, color, lbl in line_specs_realistic[:step]:
        add_threshold_line(ax, shift=shift, label=lbl, color=color, linewidth=1)
    add_combined_legend(ax, loc="upper left")
    im_pr = fig_to_image(fig)
    for _ in range(PARALLEL_REAL_HOLD):
        frames_pr.append(im_pr.copy())

save_gif(frames_pr, "28_parallel_lines_additive_realistic.gif", duration=PARALLEL_REAL_MS)

# --- Line-level proportions (realistic): one GIF, lines appear in order; first line counts with grow/shrink. ---
targets = [1, 2, 3]
line_colors = ["#1f77b4", "#17becf", "#2ca02c"]
prop = []
counts = []
for value in targets:
    m = z_real == value
    prop.append(float(y_real[m].mean()) if m.any() else 0.0)
    counts.append(int(m.sum()))

frames = []


def _prop_axes_backdrop(ax):
    """Full realistic plane: limits, black threshold, all students, axis labels/ticks + legend."""
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.grid(alpha=0.12)
    add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
    draw_dataset(
        ax, study_real, exam_real, y_real, mask=np.ones(n_r, dtype=bool), alpha=0.95
    )
    add_combined_legend(ax, loc="upper left")


def _prob_xy_for_line(shift, x_hint=5.35):
    y_line = x_hint - shift
    return x_hint + 0.12, y_line + 0.28


def _emit_hold(img, n):
    for _ in range(n):
        frames.append(img.copy())


lines_pairs = []

for _ in range(10):
    fig, ax = plt.subplots(figsize=(8, 6))
    _prop_axes_backdrop(ax)
    frames.append(fig_to_image(fig))

for ti, (value, lcol) in enumerate(zip(targets, line_colors)):
    lines_pairs.append((value, lcol))

    for _ in range(5):
        fig, ax = plt.subplots(figsize=(8, 6))
        _prop_axes_backdrop(ax)
        for sh, col in lines_pairs:
            add_threshold_line(ax, shift=sh, label=None, color=col, linewidth=1)
        frames.append(fig_to_image(fig))

    if value == 1:
        idx1 = np.where(z_real == 1)[0]
        idx1 = idx1[np.argsort(study_real[idx1], kind="mergesort")]
        n1 = len(idx1)
        for j in range(0, n1 + 1):
            grow_states = (True, False) if j > 0 else (True,)
            for grow in grow_states:
                fig, ax = plt.subplots(figsize=(8, 6))
                _prop_axes_backdrop(ax)
                for sh, col in lines_pairs:
                    add_threshold_line(ax, shift=sh, label=None, color=col, linewidth=1)
                for k, ii in enumerate(idx1):
                    if k >= j:
                        break
                    last = k == j - 1
                    zm = 0.26 if last and grow else 0.16
                    add_outcome_icon(ax, study_real[ii], exam_real[ii], passed=bool(y_real[ii]), zoom=zm, alpha=0.95)
                if j > 0:
                    pos = int(y_real[idx1[:j]].sum())
                    tx, ty = _prob_xy_for_line(1)
                    ax.text(tx, ty, f"{pos}/{j}", fontsize=NOTE_SIZE, color="black")
                im = fig_to_image(fig)
                frames.append(im)
                _emit_hold(im, 1 if grow else 0)
        p1, c1 = prop[ti], counts[ti]
        pos1 = int(round(p1 * c1))
        for _ in range(10):
            fig, ax = plt.subplots(figsize=(8, 6))
            _prop_axes_backdrop(ax)
            for sh, col in lines_pairs:
                add_threshold_line(ax, shift=sh, label=None, color=col, linewidth=1)
            tx, ty = _prob_xy_for_line(1)
            ax.text(tx, ty, f"{pos1}/{c1} = {100 * p1:.0f}%", fontsize=NOTE_SIZE, color="black")
            frames.append(fig_to_image(fig))
    else:
        idxv = np.where(z_real == value)[0]
        idxv = idxv[np.argsort(study_real[idxv], kind="mergesort")]
        for _ in range(8):
            fig, ax = plt.subplots(figsize=(8, 6))
            _prop_axes_backdrop(ax)
            for sh, col in lines_pairs:
                add_threshold_line(ax, shift=sh, label=None, color=col, linewidth=1)
            pv, cv = prop[ti], counts[ti]
            posv = int(round(pv * cv))
            tx, ty = _prob_xy_for_line(value)
            ax.text(tx, ty, f"{posv}/{cv} = {100 * pv:.0f}%", fontsize=NOTE_SIZE, color="black")
            frames.append(fig_to_image(fig))

save_gif(frames, "29_proportions_60_80_100.gif", duration=SC5_MS)

print("ST-EL=+1 pass rate:", prop[0])
print("ST-EL=+2 pass rate:", prop[1])
print("ST-EL=+3 pass rate:", prop[2])


ST-EL=+1 pass rate: 0.6
ST-EL=+2 pass rate: 0.75
ST-EL=+3 pass rate: 1.0


## Scene 6 - Student between 60% and 80%

Script alignment: a student with `ST - EL` between **+1** and **+2** should map to a probability between **0.6** and **0.8**.

`30_between_60_and_80.png` is the **study vs exam** plane (axis labels + legend like other dataset views): **`ST - EL = 0`** labeled in black; **`ST - EL = 1` / `2`** use the same blue/cyan as **`29_proportions_60_80_100.gif`**; **black marker** at **(3.5, 2)** (`z = 1.5`).


In [160]:
# Scene 6 — "between" student: ST–EL plane with z = +1 / +2 guides (same colors as proportions `29`) + black marker at (3.5, 2); axis labels + threshold legend.
z_anchor = np.array([1.0, 2.0, 3.0])
p_anchor = np.array([0.6, 0.8, 1.0])

query_z = 1.5
query_p = np.interp(query_z, z_anchor, p_anchor)

fig, ax = plt.subplots(figsize=(8, 6))
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
ax.grid(alpha=0.12)
add_threshold_line(ax, shift=0, label="ST - EL = 0", color="black", linewidth=1)
add_threshold_line(ax, shift=1, label=None, color="#1f77b4", linewidth=1)
add_threshold_line(ax, shift=2, label=None, color="#17becf", linewidth=1)
draw_dataset(ax, study_real, exam_real, y_real, alpha=0.32)
ax.plot(
    [3.5],
    [2.0],
    marker="o",
    markersize=11,
    markerfacecolor="black",
    markeredgecolor="black",
    linestyle="none",
    zorder=15,
)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "30_between_60_and_80.png")

print(f"Interpolated probability for ST-EL={query_z}: {query_p:.2f}")


Interpolated probability for ST-EL=1.5: 0.70


## Scene 6b - Normalization vs order (bridge into $2^x$)

Script alignment (`script.md` ~138–185): **realistic** plane (left) with **ST − EL = 0** threshold; **light green / dark green** (positive $z$) and **light red** ($z<0$) segments are perpendicular distances to that line. **Right:** vertical **ST − EL** bars at **A** / **B** ($y\in[-2,7]$, integer ticks). **B** anchored at **(6,2)**; **A** at **$z=+1$** pass and **$z=-1$** fail, chosen with separable labels and **not** on another student’s perpendicular segment to the threshold. GIFs step through share formulas **$1/(1{+}4)\!\to\!1/5\!\to\!20\%$** (and the **$(-1,4)$** / **$|z|$** variants). **`35`/`36`** walk integer **$z\in[-3,3]$** on the bars (on-line **$z=0$** uses a **black circle** on the plane). Same **red–white–green** colormap as **sigmoid 3D** (Scene 8) before **`38_exponential_mapping.gif`**.

Exports **`31`–`36`** (see Scene 9 checklist).

In [183]:
# Scene 6b — normalization bridge (exports 31–36): greens/reds, formula steps, exp bar colormap
NORM_MS = 42
NORM_DUO_FIGSIZE = (11.2, 5.35)
COL_POS_LO = "#b8e8b8"
COL_POS_HI = "#1a6b1a"
COL_NEG_LO = "#f5bcbc"
N_BAR_YLIM = (-2, 7)
N_BAR_YTICKS = np.arange(-2, 8)
N_BAR_XLIM = (-0.75, 1.75)
N_STACK_X = 0.5
N_STACK_WIDTH = 0.5 * (N_BAR_XLIM[1] - N_BAR_XLIM[0])
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import patheffects

# Stacked-bar formula labels (32–34): large type, always above bars; white text stroked for contrast on translucent red.
NORM_STACK_LABEL_FS = NOTE_SIZE + 14
NORM_STACK_LABEL_Z = 30
NORM_STACK_WHITE_PE = [patheffects.withStroke(linewidth=2.5, foreground="black")]

# Match Scene 8 sigmoid 3D `CMAP`: red (low) -> white (mid) -> green (high)
_EXPC = [0, 0.5, 1.0]
_EXPK = ["red", "white", "green"]
_EXPN = plt.Normalize(min(_EXPC), max(_EXPC))
_EXPT = list(zip(map(_EXPN, _EXPC), _EXPK))
_EXP_SIG_CMAP = LinearSegmentedColormap.from_list("exp_sig_rwg", _EXPT, 100)


def _exp_bar_facecolors(z_vals):
    out = []
    for z in z_vals:
        # Match sigmoid 3D semantics: z=0 -> white; z<0 redder, z>0 greener.
        t = float(z) / 6.0 + 0.5
        out.append(_EXP_SIG_CMAP(np.clip(t, 0.0, 1.0)))
    return out


def _norm_frame_png(fig):
    # Full figure bbox (no tight crop) so every GIF frame has identical pixel size.
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=140, bbox_inches=None)
    buf.seek(0)
    im = Image.open(buf).convert("RGB").copy()
    buf.close()
    return im


def _threshold_foot(s, e, shift):
    t = (s + e + shift) / 2.0
    return t, t - shift


def _pick_realistic(s_target, e_target):
    m = np.isclose(study_real, s_target) & np.isclose(exam_real, e_target)
    idx = int(np.flatnonzero(m)[0])
    return idx, float(study_real[idx]), float(exam_real[idx]), float(z_real[idx]), int(y_real[idx])


def _draw_plane_base(ax):
    # Clear each frame so repeated threshold draws do not stack (linewidth creep in GIFs).
    ax.clear()
    draw_dataset(ax, study_real, exam_real, y_real, alpha=0.22)
    add_threshold_line(
        ax,
        shift=midpoint_shift,
        label="ST - EL = 0",
        color="black",
        linewidth=1,
        x_range=tuple(ax.get_xlim()),
    )
    add_combined_legend(ax, loc="upper left")


def _draw_student_distance(ax, s, e, y, color):
    fs, fe = _threshold_foot(s, e, midpoint_shift)
    ax.plot([s, fs], [e, fe], color=color, linewidth=2.8, solid_capstyle="round", zorder=4)
    add_outcome_icon(ax, s, e, passed=bool(y), alpha=0.98, zoom=0.16)


def _norm6b_on_perp_open(px, py, qx, qy, shift, eps=1e-5):
    """True if Q lies strictly between P and the foot of the perpendicular from P to ST-EL = shift."""
    P = np.array([px, py], dtype=float)
    Q = np.array([qx, qy], dtype=float)
    t = (px + py + shift) / 2.0
    F = np.array([t, t - shift], dtype=float)
    v = F - P
    nv = float(np.hypot(v[0], v[1]))
    if nv < eps:
        return False
    w = Q - P
    if abs(w[0] * v[1] - w[1] * v[0]) > eps * nv:
        return False
    tv = float(np.dot(v, v))
    if tv < eps * eps:
        return False
    tparam = float(np.dot(w, v) / tv)
    return eps < tparam < 1.0 - eps


def _draw_barhint_plane_marker(ax, s, e, y, z, color):
    fs, fe = _threshold_foot(s, e, midpoint_shift)
    ax.plot([s, fs], [e, fe], color=color, linewidth=2.8, solid_capstyle="round", zorder=4)
    if np.isclose(z, 0.0):
        ax.plot(
            [s],
            [e],
            linestyle="none",
            marker="o",
            markersize=11,
            markerfacecolor="black",
            markeredgecolor="black",
            zorder=8,
        )
    else:
        add_outcome_icon(ax, s, e, passed=bool(y), alpha=0.98, zoom=0.16)


def _norm6b_pick_index_for_z(zt, want_y, ref_s, ref_e):
    idxs = np.flatnonzero(np.isclose(z_real, zt, atol=1e-9) & (y_real == int(want_y)))
    if not idxs.size:
        idxs = np.flatnonzero(np.isclose(z_real, zt, atol=1e-9))
    sh = float(midpoint_shift)
    best_i = None
    best_key = None
    relax_i = None
    relax_key = None
    for i in idxs:
        s = float(study_real[i])
        ee = float(exam_real[i])
        ok = (not _norm6b_on_perp_open(ref_s, ref_e, s, ee, sh)) and (not _norm6b_on_perp_open(s, ee, ref_s, ref_e, sh))
        md = float(np.hypot(s - ref_s, ee - ref_e))
        key = (md, -s)
        if relax_i is None or key > relax_key:
            relax_key, relax_i = key, int(i)
        if ok and (best_i is None or key > best_key):
            best_key, best_i = key, int(i)
    return int(best_i if best_i is not None else relax_i)


def _draw_left_progress(ax, s_a, e_a, y_a, c_a, s_b, e_b, y_b, c_b, phase):
    # phase: 0 none, 1 A only, 2 A+B (B then A so A's perpendicular stays on top when B is on ST-EL=0).
    _draw_plane_base(ax)
    if phase >= 2:
        _draw_student_distance(ax, s_b, e_b, y_b, c_b)
    if phase == 1:
        _draw_student_distance(ax, s_a, e_a, y_a, c_a)
    elif phase >= 2:
        _draw_student_distance(ax, s_a, e_a, y_a, c_a)


def _style_bar_axis(ax, stacked=False, ylabel=None, z_ylim=None):
    ylab = ylabel if ylabel is not None else "ST - EL"
    if z_ylim is None:
        ax.set_ylim(*N_BAR_YLIM)
        ax.set_yticks(N_BAR_YTICKS)
    else:
        ax.set_ylim(z_ylim[0], z_ylim[1])
        ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=8, prune="lower"))
    ax.set_ylabel(ylab, fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.set_xlabel("")
    ax.tick_params(axis="both", labelsize=FONT_SIZE)
    ax.grid(axis="y", alpha=0.22)
    ax.axhline(0.0, color="black", linewidth=1.0, linestyle=":")
    ax.set_xlim(*N_BAR_XLIM)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["A", "B"])


def _draw_twin_bars(ax, z_a, z_b, c_a, c_b, phase, alpha_a=1.0, ylabel=None, z_ylim=None):
    # phase: 0 none, 1 A only, 2 A+B
    ax.clear()
    if phase >= 1:
        ax.bar([0], [z_a], width=0.52, color=[c_a], edgecolor="black", linewidth=0.9, alpha=alpha_a)
    if phase >= 2:
        ax.bar([1], [z_b], width=0.52, color=[c_b], edgecolor="black", linewidth=0.9)
    _style_bar_axis(ax, stacked=False, ylabel=ylabel, z_ylim=z_ylim)


def _draw_pos_stack_labels(ax_r, stage):
    h1, h2 = 1.0, 4.0
    fs = NORM_STACK_LABEL_FS
    zz = NORM_STACK_LABEL_Z
    if stage == 1:
        ax_r.text(N_STACK_X, 0.5 * h1, r"$1/(1+4)$", ha="center", va="center", fontsize=fs, fontweight="bold", color="#0a3d0a", zorder=zz)
        ax_r.text(
            N_STACK_X,
            h1 + 0.5 * h2,
            r"$4/(1+4)$",
            ha="center",
            va="center",
            fontsize=fs,
            fontweight="bold",
            color="#ffffff",
            path_effects=NORM_STACK_WHITE_PE,
            zorder=zz,
        )
    elif stage == 2:
        ax_r.text(N_STACK_X, 0.5 * h1, r"$\frac{1}{5}$", ha="center", va="center", fontsize=fs, fontweight="bold", color="#0a3d0a", zorder=zz)
        ax_r.text(
            N_STACK_X,
            h1 + 0.5 * h2,
            r"$\frac{4}{5}$",
            ha="center",
            va="center",
            fontsize=fs,
            fontweight="bold",
            color="#ffffff",
            path_effects=NORM_STACK_WHITE_PE,
            zorder=zz,
        )
    elif stage == 3:
        ax_r.text(N_STACK_X, 0.5 * h1, "20%", ha="center", va="center", fontsize=fs, fontweight="bold", color="#0a3d0a", zorder=zz)
        ax_r.text(
            N_STACK_X,
            h1 + 0.5 * h2,
            "80%",
            ha="center",
            va="center",
            fontsize=fs,
            fontweight="bold",
            color="#ffffff",
            path_effects=NORM_STACK_WHITE_PE,
            zorder=zz,
        )


def _draw_neg_stack_labels(ax_r, stage):
    zn, zp = -1.0, 4.0
    fs = NORM_STACK_LABEL_FS
    zz = NORM_STACK_LABEL_Z
    kw = dict(ha="center", va="center", fontsize=fs, fontweight="bold", color="#ffffff", alpha=1.0, path_effects=NORM_STACK_WHITE_PE, zorder=zz)
    if stage == 1:
        ax_r.text(N_STACK_X, 0.5 * zn, r"$\frac{-1}{-1+4}$", **kw)
        ax_r.text(N_STACK_X, zn + 0.5 * zp, r"$\frac{4}{-1+4}$", **kw)
    elif stage == 2:
        ax_r.text(N_STACK_X, 0.5 * zn, r"$-\frac{1}{3}$", **kw)
        ax_r.text(N_STACK_X, zn + 0.5 * zp, r"$\frac{4}{3}$", **kw)


def _with_pos_stack(ax_r, stage):
    h1, h2 = 1.0, 4.0
    ax_r.clear()
    ax_r.bar([N_STACK_X], [h1], width=N_STACK_WIDTH, bottom=0.0, color=COL_POS_LO, edgecolor="black", linewidth=0.9)
    ax_r.bar([N_STACK_X], [h2], width=N_STACK_WIDTH, bottom=h1, color=COL_POS_HI, edgecolor="black", linewidth=0.9)
    _style_bar_axis(ax_r, stacked=True)
    if stage > 0:
        _draw_pos_stack_labels(ax_r, stage)


def _with_neg_stack(ax_r, stage):
    zn, zp = -1.0, 4.0
    ax_r.clear()
    # Draw green first, then translucent red on top for visibility of overlap.
    ax_r.bar([N_STACK_X], [zp], width=N_STACK_WIDTH, bottom=zn, color=COL_POS_HI, edgecolor="black", linewidth=0.9, zorder=2)
    ax_r.bar([N_STACK_X], [zn], width=N_STACK_WIDTH, bottom=0.0, color=COL_NEG_LO, edgecolor="black", linewidth=0.9, zorder=4, alpha=0.62)
    _style_bar_axis(ax_r, stacked=True)
    if stage > 0:
        _draw_neg_stack_labels(ax_r, stage)


def _with_abs_stack(ax_r, stage, ylabel=None):
    h1, h2 = abs(ZAN), ZB4
    ax_r.clear()
    ax_r.bar([N_STACK_X], [h1], width=N_STACK_WIDTH, bottom=0.0, color=COL_POS_LO, edgecolor="black", linewidth=0.9)
    ax_r.bar([N_STACK_X], [h2], width=N_STACK_WIDTH, bottom=h1, color=COL_POS_HI, edgecolor="black", linewidth=0.9)
    _style_bar_axis(ax_r, stacked=True, ylabel=ylabel)
    fs = NORM_STACK_LABEL_FS
    zz = NORM_STACK_LABEL_Z
    if stage == 1:
        ax_r.text(N_STACK_X, 0.5 * h1, r"$1/(1+4)$", ha="center", va="center", fontsize=fs, fontweight="bold", color="#0a3d0a", zorder=zz)
        ax_r.text(
            N_STACK_X,
            h1 + 0.5 * h2,
            r"$4/(1+4)$",
            ha="center",
            va="center",
            fontsize=fs,
            fontweight="bold",
            color="#ffffff",
            path_effects=NORM_STACK_WHITE_PE,
            zorder=zz,
        )
    elif stage == 2:
        ax_r.text(N_STACK_X, 0.5 * h1, r"$\frac{1}{5}$", ha="center", va="center", fontsize=fs, fontweight="bold", color="#0a3d0a", zorder=zz)
        ax_r.text(
            N_STACK_X,
            h1 + 0.5 * h2,
            r"$\frac{4}{5}$",
            ha="center",
            va="center",
            fontsize=fs,
            fontweight="bold",
            color="#ffffff",
            path_effects=NORM_STACK_WHITE_PE,
            zorder=zz,
        )
    elif stage == 3:
        ax_r.text(N_STACK_X, 0.5 * h1, "20%", ha="center", va="center", fontsize=fs, fontweight="bold", color="#0a3d0a", zorder=zz)
        ax_r.text(
            N_STACK_X,
            h1 + 0.5 * h2,
            "80%",
            ha="center",
            va="center",
            fontsize=fs,
            fontweight="bold",
            color="#ffffff",
            path_effects=NORM_STACK_WHITE_PE,
            zorder=zz,
        )


_, SB4, EB4, ZB4, YB4 = _pick_realistic(6, 2)
_i_pos = _norm6b_pick_index_for_z(1.0, 1, SB4, EB4)
SA1, EA1, ZA1, YA1 = float(study_real[_i_pos]), float(exam_real[_i_pos]), float(z_real[_i_pos]), int(y_real[_i_pos])
_i_neg = _norm6b_pick_index_for_z(-1.0, 0, SB4, EB4)
SAN, EAN, ZAN, YAN = float(study_real[_i_neg]), float(exam_real[_i_neg]), float(z_real[_i_neg]), int(y_real[_i_neg])
assert np.isclose(ZA1, 1) and np.isclose(ZB4, 4) and np.isclose(ZAN, -1)
assert YA1 == 1 and YB4 == 1 and YAN == 0

# --- 31 ---
fig, (axl, axr) = plt.subplots(1, 2, figsize=NORM_DUO_FIGSIZE, gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28})
_draw_left_progress(axl, SA1, EA1, YA1, COL_POS_LO, SB4, EB4, YB4, COL_POS_HI, phase=2)
_draw_twin_bars(axr, ZA1, ZB4, COL_POS_LO, COL_POS_HI, phase=2)
fig.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
save_fig(fig, "31_norm_positive_two_bars.png")

# --- 32 GIF ---
frames_nb = []
fig0, (ax0_l, ax0_r) = plt.subplots(1, 2, figsize=NORM_DUO_FIGSIZE, gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28})
fig0.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
for _ in range(4):
    _draw_left_progress(ax0_l, SA1, EA1, YA1, COL_POS_LO, SB4, EB4, YB4, COL_POS_HI, phase=0)
    _draw_twin_bars(ax0_r, ZA1, ZB4, COL_POS_LO, COL_POS_HI, phase=0)
    frames_nb.append(_norm_frame_png(fig0))
for _ in range(5):
    _draw_left_progress(ax0_l, SA1, EA1, YA1, COL_POS_LO, SB4, EB4, YB4, COL_POS_HI, phase=1)
    _draw_twin_bars(ax0_r, ZA1, ZB4, COL_POS_LO, COL_POS_HI, phase=1)
    frames_nb.append(_norm_frame_png(fig0))
for _ in range(5):
    _draw_left_progress(ax0_l, SA1, EA1, YA1, COL_POS_LO, SB4, EB4, YB4, COL_POS_HI, phase=2)
    _draw_twin_bars(ax0_r, ZA1, ZB4, COL_POS_LO, COL_POS_HI, phase=2)
    frames_nb.append(_norm_frame_png(fig0))
for st in (0, 1, 2, 3):
    nrep = 6 if st == 3 else 5
    for _ in range(nrep):
        _draw_left_progress(ax0_l, SA1, EA1, YA1, COL_POS_LO, SB4, EB4, YB4, COL_POS_HI, phase=2)
        _with_pos_stack(ax0_r, st)
        frames_nb.append(_norm_frame_png(fig0))
plt.close(fig0)
save_gif(frames_nb, "32_norm_positive_stack.gif", duration=NORM_MS)

# --- 33 GIF ---
frames_neg = []
f1, (ax1_l, ax1_r) = plt.subplots(1, 2, figsize=NORM_DUO_FIGSIZE, gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28})
f1.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
for _ in range(4):
    _draw_left_progress(ax1_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=0)
    _draw_twin_bars(ax1_r, ZAN, ZB4, COL_NEG_LO, COL_POS_HI, phase=0)
    frames_neg.append(_norm_frame_png(f1))
for _ in range(5):
    _draw_left_progress(ax1_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=1)
    _draw_twin_bars(ax1_r, ZAN, ZB4, COL_NEG_LO, COL_POS_HI, phase=1, alpha_a=0.62)
    frames_neg.append(_norm_frame_png(f1))
for _ in range(5):
    _draw_left_progress(ax1_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=2)
    _draw_twin_bars(ax1_r, ZAN, ZB4, COL_NEG_LO, COL_POS_HI, phase=2, alpha_a=0.62)
    frames_neg.append(_norm_frame_png(f1))
for st in (0, 1, 2):
    nrep = 7 if st == 2 else 5
    for _ in range(nrep):
        _draw_left_progress(ax1_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=2)
        _with_neg_stack(ax1_r, st)
        frames_neg.append(_norm_frame_png(f1))
plt.close(f1)
save_gif(frames_neg, "33_norm_negative_invalid.gif", duration=NORM_MS)

# --- 34 GIF ---
frames_abs = []
f2, (ax2_l, ax2_r) = plt.subplots(1, 2, figsize=NORM_DUO_FIGSIZE, gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28})
f2.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
# signed reveal: none -> A -> A+B
for _ in range(4):
    _draw_left_progress(ax2_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=0)
    _draw_twin_bars(ax2_r, ZAN, ZB4, COL_NEG_LO, COL_POS_HI, phase=0)
    frames_abs.append(_norm_frame_png(f2))
for _ in range(5):
    _draw_left_progress(ax2_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=1)
    _draw_twin_bars(ax2_r, ZAN, ZB4, COL_NEG_LO, COL_POS_HI, phase=1, alpha_a=0.62)
    frames_abs.append(_norm_frame_png(f2))
for _ in range(5):
    _draw_left_progress(ax2_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=2)
    _draw_twin_bars(ax2_r, ZAN, ZB4, COL_NEG_LO, COL_POS_HI, phase=2, alpha_a=0.62)
    frames_abs.append(_norm_frame_png(f2))
# absolute twin reveal: none -> A -> A+B
for _ in range(4):
    _draw_left_progress(ax2_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=0)
    _draw_twin_bars(ax2_r, abs(ZAN), ZB4, COL_POS_LO, COL_POS_HI, phase=0, ylabel=r"$|\mathrm{ST}-\mathrm{EL}|$")
    frames_abs.append(_norm_frame_png(f2))
for _ in range(5):
    _draw_left_progress(ax2_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=1)
    _draw_twin_bars(ax2_r, abs(ZAN), ZB4, COL_POS_LO, COL_POS_HI, phase=1, ylabel=r"$|\mathrm{ST}-\mathrm{EL}|$")
    frames_abs.append(_norm_frame_png(f2))
for _ in range(5):
    _draw_left_progress(ax2_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=2)
    _draw_twin_bars(ax2_r, abs(ZAN), ZB4, COL_POS_LO, COL_POS_HI, phase=2, ylabel=r"$|\mathrm{ST}-\mathrm{EL}|$")
    frames_abs.append(_norm_frame_png(f2))
for pst in (1, 2, 3):
    nrep = 7 if pst == 3 else 5
    for _ in range(nrep):
        _draw_left_progress(ax2_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=2)
        _with_abs_stack(ax2_r, pst, ylabel=r"$|\mathrm{ST}-\mathrm{EL}|$")
        frames_abs.append(_norm_frame_png(f2))
plt.close(f2)
save_gif(frames_abs, "34_norm_abs_pitfall.gif", duration=NORM_MS)

# --- 35: GIF — |z| bars (same cadence as 36); negative z shows signed bar then flips to |z|
ZS_BAR_HINT = np.arange(-3, 4, dtype=float)
BARHINT_XLIM = (-3.9, 3.9)
BARHINT_BAR_W = 0.34
ABS_F_YLIM = (-3.9, 3.9)
ABS_F_YTICKS = np.arange(-3, 5)
_ABS_F_YLABEL = r"$|\mathrm{ST}-\mathrm{EL}|$"
BARHINT_NONZERO_ORDER = (-3, -2, -1, 1, 2, 3)


def _build_barhint_points(zs_vals):
    sh = float(midpoint_shift)
    want = {int(z) for z in zs_vals}
    chosen_s = [4.0]
    chosen_e = [4.0]
    out = {0: (4.0, 4.0, 1)}
    for zt in BARHINT_NONZERO_ORDER:
        if zt not in want:
            continue
        want_y = 1 if zt > 0 else 0
        idxs = np.flatnonzero(np.isclose(z_real, float(zt), atol=1e-9) & (y_real == want_y))
        if not idxs.size:
            idxs = np.flatnonzero(np.isclose(z_real, float(zt), atol=1e-9))
        best_i = None
        best_key = None
        relax_i = None
        relax_key = None
        for i in idxs:
            s = float(study_real[i])
            ee = float(exam_real[i])
            y = int(y_real[i])
            ok = True
            for sj, ej in zip(chosen_s, chosen_e):
                if _norm6b_on_perp_open(sj, ej, s, ee, sh) or _norm6b_on_perp_open(s, ee, sj, ej, sh):
                    ok = False
                    break
            md = min(np.hypot(s - sj, ee - ej) for sj, ej in zip(chosen_s, chosen_e))
            key = (md, -s)
            if relax_i is None or key > relax_key:
                relax_key, relax_i = key, int(i)
            if ok and (best_i is None or key > best_key):
                best_key, best_i = key, int(i)
        use_i = int(best_i if best_i is not None else relax_i)
        s = float(study_real[use_i])
        ee = float(exam_real[use_i])
        y = int(y_real[use_i])
        out[int(zt)] = (s, ee, y)
        chosen_s.append(s)
        chosen_e.append(ee)
    return out


BARHINT_POINTS = _build_barhint_points(ZS_BAR_HINT)


def _style_abs_f_axis(ax):
    ax.set_xlim(*BARHINT_XLIM)
    ax.set_xticks(np.arange(-3, 4, dtype=int))
    ax.set_xlabel("ST - EL", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.set_ylim(*ABS_F_YLIM)
    ax.set_yticks(ABS_F_YTICKS)
    ax.set_ylabel(_ABS_F_YLABEL, fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.tick_params(axis="both", labelsize=FONT_SIZE)
    ax.grid(axis="y", alpha=0.22)
    ax.axhline(0.0, color="black", linewidth=1.0, linestyle=":")


def _draw_abs_f_bars(ax_r, specs):
    ax_r.clear()
    for z, mode in specs:
        col = _exp_bar_facecolors([float(z)])[0]
        if mode == "signed" and z < 0:
            ax_r.bar([z], [z], width=BARHINT_BAR_W, bottom=0.0, color=col, edgecolor="black", linewidth=0.85, zorder=3)
        else:
            h = float(abs(z))
            ax_r.bar([z], [h], width=BARHINT_BAR_W, bottom=0.0, color=col, edgecolor="black", linewidth=0.85, zorder=3)
    _style_abs_f_axis(ax_r)


def _draw_barhint_left(ax_l, zs_show, points_map):
    _draw_plane_base(ax_l)
    for z in zs_show:
        zi = int(z)
        s, e, y = points_map[zi]
        col = _exp_bar_facecolors([float(zi)])[0]
        _draw_barhint_plane_marker(ax_l, s, e, y, float(zi), col)


frames_abs_f = []
fig_af, (ax_af_l, ax_af_r) = plt.subplots(
    1,
    2,
    figsize=NORM_DUO_FIGSIZE,
    gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28},
)
fig_af.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
for k in range(len(ZS_BAR_HINT)):
    z_new = float(ZS_BAR_HINT[k])
    if z_new < 0:
        for _ in range(4):
            zs_show = list(ZS_BAR_HINT[: k + 1])
            specs = [(float(ZS_BAR_HINT[j]), "exp") for j in range(k)] + [(z_new, "signed")]
            _draw_barhint_left(ax_af_l, zs_show, BARHINT_POINTS)
            _draw_abs_f_bars(ax_af_r, specs)
            frames_abs_f.append(_norm_frame_png(fig_af))
        for _ in range(4):
            zs_show = list(ZS_BAR_HINT[: k + 1])
            specs = [(float(ZS_BAR_HINT[j]), "exp") for j in range(k + 1)]
            _draw_barhint_left(ax_af_l, zs_show, BARHINT_POINTS)
            _draw_abs_f_bars(ax_af_r, specs)
            frames_abs_f.append(_norm_frame_png(fig_af))
    else:
        for _ in range(5):
            zs_show = list(ZS_BAR_HINT[: k + 1])
            specs = [(float(ZS_BAR_HINT[j]), "exp") for j in range(k + 1)]
            _draw_barhint_left(ax_af_l, zs_show, BARHINT_POINTS)
            _draw_abs_f_bars(ax_af_r, specs)
            frames_abs_f.append(_norm_frame_png(fig_af))
plt.close(fig_af)
save_gif(frames_abs_f, "35_norm_abs_bar_hint.gif", duration=NORM_MS)

# --- 36: GIF — 2^z with dataset + perpendicular distances; negative z shows signed bar then flips to f(z)=2^z
ZS_EXP = ZS_BAR_HINT
EXP_XLIM = BARHINT_XLIM
EXP_YLIM = (-3.9, 9.5)
EXP_YTICKS = np.arange(-3, 10)
EXP_BAR_W = BARHINT_BAR_W
_EXP_YLABEL = r"$f(\mathrm{ST}-\mathrm{EL})$"
EXP_POINTS = BARHINT_POINTS


def _style_exp_f_axis(ax):
    ax.set_xlim(*EXP_XLIM)
    ax.set_xticks(np.arange(-3, 4, dtype=int))
    ax.set_xlabel("ST - EL", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.set_ylim(*EXP_YLIM)
    ax.set_yticks(EXP_YTICKS)
    ax.set_ylabel(_EXP_YLABEL, fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.tick_params(axis="both", labelsize=FONT_SIZE)
    ax.grid(axis="y", alpha=0.22)
    ax.axhline(0.0, color="black", linewidth=1.0, linestyle=":")


def _draw_exp_f_bars(ax_r, specs):
    ax_r.clear()
    for z, mode in specs:
        col = _exp_bar_facecolors([float(z)])[0]
        if mode == "signed" and z < 0:
            ax_r.bar([z], [z], width=EXP_BAR_W, bottom=0.0, color=col, edgecolor="black", linewidth=0.85, zorder=3)
        else:
            h = float(np.power(2.0, z))
            ax_r.bar([z], [h], width=EXP_BAR_W, bottom=0.0, color=col, edgecolor="black", linewidth=0.85, zorder=3)
    _style_exp_f_axis(ax_r)


def _draw_exp_left(ax_l, zs_show):
    _draw_plane_base(ax_l)
    for z in zs_show:
        s, e, y = EXP_POINTS[int(z)]
        col = _exp_bar_facecolors([float(z)])[0]
        _draw_student_distance(ax_l, s, e, y, col)


frames_eb = []
fig_e, (ax_e_l, ax_e_r) = plt.subplots(
    1,
    2,
    figsize=NORM_DUO_FIGSIZE,
    gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28},
)
fig_e.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
for k in range(len(ZS_EXP)):
    z_new = float(ZS_EXP[k])
    if z_new < 0:
        for _ in range(4):
            zs_show = list(ZS_EXP[: k + 1])
            specs = [(float(ZS_EXP[j]), "exp") for j in range(k)] + [(z_new, "signed")]
            _draw_exp_left(ax_e_l, zs_show)
            _draw_exp_f_bars(ax_e_r, specs)
            frames_eb.append(_norm_frame_png(fig_e))
        for _ in range(4):
            zs_show = list(ZS_EXP[: k + 1])
            specs = [(float(ZS_EXP[j]), "exp") for j in range(k + 1)]
            _draw_exp_left(ax_e_l, zs_show)
            _draw_exp_f_bars(ax_e_r, specs)
            frames_eb.append(_norm_frame_png(fig_e))
    else:
        for _ in range(5):
            zs_show = list(ZS_EXP[: k + 1])
            specs = [(float(ZS_EXP[j]), "exp") for j in range(k + 1)]
            _draw_exp_left(ax_e_l, zs_show)
            _draw_exp_f_bars(ax_e_r, specs)
            frames_eb.append(_norm_frame_png(fig_e))
plt.close(fig_e)
save_gif(frames_eb, "36_norm_exp_bar_hint.gif", duration=NORM_MS)


## Scene 7 - Exponential step-by-step ($2^x$, step size 1 in $x$)

Script alignment (Chapter 1 script):
- moving **one unit left** in $x$ (so $\Delta x=-1$) must shrink the positive output while leaving room for all negatives
- the consistent rule is **multiply by $\frac{1}{2}$** each step — that is exactly **$2^x$** on integer steps

- **Scene 6b (`31`–`36`)** motivates exponentials: **`35`/`36`** step **$z\in[-3,3]$** on the bars (**$z=0$** on the plane is a **black circle**); **`35`** uses **$|z|$** heights; **`36`** uses **$2^{z}$** (strictly positive, increasing in $z$), same **red–white–green** colormap as sigmoid **3D**

`37_exponential_point_guides.gif` introduces a single point with dotted coordinate guides: first the **vertical** guide to its $x$ tick, then the **horizontal** guide to its $y$ tick.

`38_exponential_mapping.gif` is the local $2^x$ panel: **dotted** blue curve (**linewidth 1**), **blue** point marker; each step shows **vertical segment first**, then its label, then the next horizontal segment (label cleared).

`39_step_left_drop_one_to_zero.png` snapshots one step left with a full output drop of **1** to zero. `40_step_two_drop_half_to_zero.png` snapshots the second step with a drop of **1/2** (instead of **1/4**) to zero.

`41_exponential_transition.gif` continues from the **final frame of `38_exponential_mapping.gif`**: stairs vanish, the **$2^x$** trace becomes **solid** (**lw 2**), then the window **opens** to $x\in(-4.5,6)$ (with matching $y$-scale). Then **`42_exponential_2x_legend.png` / `43_exponential_ex_legend.png`** are static **$(-4.5,6)$** plots with legends $f(x)=2^x$ and $f(x)=e^x$ (same blue curve color). **`44_norm_exp_positive_softmax.gif`** / **`45_norm_exp_negative_softmax.gif`** replay the **32**/**33** duo-panel cadence with **$e^{\mathrm{ST}-\mathrm{EL}}$** bars, a stacked **softmax** denominator, explicit normalization fractions, then **three-decimal** masses. **`46_far_study_hours_zoom.gif`** opens on the **duo** layout: the **right** panel is an empty **ST − EL** bar axis until the **left** finishes zooming to study time $[4,120]$ and exam length $[4,110]$ with **ST $-$ EL** gaps **101** (student **A**) and **104** (student **B**), and **both** are **selected** (perpendicular guides); then **twin ST − EL** bars, **exp** bars, stacked **softmax**, and the same probability callout as **44**/**45**. **`47_softmax_threshold_and_z1.gif`** places **B** on **ST − EL = 0** and **A** at **ST − EL = 1** with the same **exp**/**softmax** beats. Scene **8** then adds **four** softmax snapshot PNGs (**`48`–`51`**), **two** **2D** sigmoid reveal GIFs (**`52`**, **`53`**), and **three** matching **2D** sigmoid legend panels (**`54`–`56`**) before the **3D** rotation exports.


In [193]:
# 38 — exponential mapping: dotted blue curve (lw=1); blue marker; stairs across → down → label → across (label cleared) …
EXP_STEP_HOLD = 5
STAIR_LW = 1
CURVE_COLOR = "#1f77b4"
CURVE_LW = 1.0

frames = []


def _strip(ax):
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("")
    ax.tick_params(labelbottom=True, labelleft=True, labelsize=FONT_SIZE)


def _v_step_label(s):
    return {1: "1/2", 2: "1/4", 3: "1/8"}.get(s, "...")


def _plot_curve(ax, xs, x_min):
    m = xs >= x_min
    ax.plot(
        xs[m],
        np.power(2.0, xs[m]),
        color=CURVE_COLOR,
        linestyle=":",
        linewidth=CURVE_LW,
        zorder=2,
    )


def _plot_full_stairs(ax, upto_s_inclusive, label_vertical_s=None):
    for s in range(1, upto_s_inclusive + 1):
        xp = -(s - 1)
        xc = -s
        yp = float(np.power(2.0, xp))
        yc = float(np.power(2.0, xc))
        ax.plot(
            [xp, xc],
            [yp, yp],
            color="black",
            linewidth=STAIR_LW,
            solid_capstyle="projecting",
            zorder=4,
        )
        ax.plot(
            [xc, xc],
            [yp, yc],
            color="black",
            linewidth=STAIR_LW,
            solid_capstyle="projecting",
            zorder=4,
        )
        if label_vertical_s is not None and s == label_vertical_s:
            vy = 0.5 * (yp + yc)
            ax.text(
                xc - 0.12,
                vy,
                _v_step_label(s),
                fontsize=ANNOT_SIZE + 2,
                ha="right",
                va="center",
                color="black",
                zorder=5,
            )


def _plot_horizontal_only(ax, s):
    xp = -(s - 1)
    xc = -s
    yp = float(np.power(2.0, xp))
    yc = float(np.power(2.0, xc))
    ax.plot(
        [xp, xc],
        [yp, yp],
        color="black",
        linewidth=STAIR_LW,
        solid_capstyle="projecting",
        zorder=4,
    )
    return xp, xc, yp, yc


def _stair_h(ax, k):
    xp = -(k - 1)
    xc = -k
    yp = float(np.power(2.0, xp))
    yc = float(np.power(2.0, xc))
    ax.plot(
        [xp, xc],
        [yp, yp],
        color="black",
        linewidth=STAIR_LW,
        solid_capstyle="projecting",
        zorder=4,
    )
    return xp, xc, yp, yc


def _stair_v(ax, k):
    xp = -(k - 1)
    xc = -k
    yp = float(np.power(2.0, xp))
    yc = float(np.power(2.0, xc))
    ax.plot(
        [xc, xc],
        [yp, yc],
        color="black",
        linewidth=STAIR_LW,
        solid_capstyle="projecting",
        zorder=4,
    )


xs_full = np.linspace(-4.5, 1.4, 520)


def _exp_snap(fig, ax):
    finalize_exponential_figure(fig)
    return fig_to_image(fig, tight_layout=False)


def _save_exp_png(fig, filename):
    fig.savefig(
        OUTPUT_DIR / filename,
        format="png",
        dpi=140,
        bbox_inches=None,
        pad_inches=SAVE_PAD_INCHES,
    )
    plt.close(fig)


# --- 37: point-only coordinate guide (vertical then horizontal dotted helpers) ---
COORD_X = 0.0
COORD_Y = 1.0
frames_coord = []
for _ in range(EXP_STEP_HOLD):
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _plot_curve(ax, xs_full, COORD_X)
    ax.scatter(
        [COORD_X],
        [COORD_Y],
        s=130,
        facecolors=CURVE_COLOR,
        edgecolors="white",
        linewidths=0.9,
        zorder=6,
        clip_on=False,
    )
    ax.set_xlim(-4.5, 1.4)
    ax.set_ylim(0, 1.15)
    ax.grid(alpha=0.2)
    _strip(ax)
    frames_coord.append(_exp_snap(fig, ax))

for _ in range(EXP_STEP_HOLD):
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _plot_curve(ax, xs_full, COORD_X)
    ax.plot([COORD_X, COORD_X], [0.0, COORD_Y], color="black", linestyle=":", linewidth=1.2, zorder=4)
    ax.scatter(
        [COORD_X],
        [COORD_Y],
        s=130,
        facecolors=CURVE_COLOR,
        edgecolors="white",
        linewidths=0.9,
        zorder=6,
        clip_on=False,
    )
    ax.set_xlim(-4.5, 1.4)
    ax.set_ylim(0, 1.15)
    ax.grid(alpha=0.2)
    _strip(ax)
    frames_coord.append(_exp_snap(fig, ax))

for _ in range(EXP_STEP_HOLD):
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _plot_curve(ax, xs_full, COORD_X)
    ax.plot([COORD_X, COORD_X], [0.0, COORD_Y], color="black", linestyle=":", linewidth=1.2, zorder=4)
    ax.plot([-4.5, COORD_X], [COORD_Y, COORD_Y], color="black", linestyle=":", linewidth=1.2, zorder=4)
    ax.scatter(
        [COORD_X],
        [COORD_Y],
        s=130,
        facecolors=CURVE_COLOR,
        edgecolors="white",
        linewidths=0.9,
        zorder=6,
        clip_on=False,
    )
    ax.set_xlim(-4.5, 1.4)
    ax.set_ylim(0, 1.15)
    ax.grid(alpha=0.2)
    _strip(ax)
    frames_coord.append(_exp_snap(fig, ax))

save_gif(frames_coord, "37_exponential_point_guides.gif", duration=70)


for _ in range(EXP_STEP_HOLD):
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _plot_curve(ax, xs_full, 0.0)
    ax.scatter(
        [0.0],
        [1.0],
        s=130,
        facecolors=CURVE_COLOR,
        edgecolors="white",
        linewidths=0.9,
        zorder=6,
        clip_on=False,
    )
    ax.set_xlim(-4.5, 1.4)
    ax.set_ylim(0, 1.15)
    ax.grid(alpha=0.2)
    _strip(ax)
    frames.append(_exp_snap(fig, ax))

for s in range(1, 7):
    xp = -(s - 1)
    yp = float(np.power(2.0, xp))
    xc = -s
    yc = float(np.power(2.0, xc))

    # 1) move across: horizontal for step s (completed through vertical s-1)
    for _ in range(EXP_STEP_HOLD):
        fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
        _plot_curve(ax, xs_full, xp)
        for k in range(1, s):
            _stair_h(ax, k)
            _stair_v(ax, k)
        ax.plot(
            [xp, xc],
            [yp, yp],
            color="black",
            linewidth=STAIR_LW,
            solid_capstyle="projecting",
            zorder=4,
        )
        ax.scatter(
            [xc],
            [yp],
            s=130,
            facecolors=CURVE_COLOR,
            edgecolors="white",
            linewidths=0.9,
            zorder=6,
            clip_on=False,
        )
        ax.set_xlim(-4.5, 1.4)
        ax.set_ylim(0, 1.15)
        ax.grid(alpha=0.2)
        _strip(ax)
        frames.append(_exp_snap(fig, ax))

    # 2) move down: vertical for step s
    for _ in range(EXP_STEP_HOLD):
        fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
        _plot_curve(ax, xs_full, xc)
        for k in range(1, s):
            _stair_h(ax, k)
            _stair_v(ax, k)
        ax.plot(
            [xp, xc],
            [yp, yp],
            color="black",
            linewidth=STAIR_LW,
            solid_capstyle="projecting",
            zorder=4,
        )
        ax.plot(
            [xc, xc],
            [yp, yc],
            color="black",
            linewidth=STAIR_LW,
            solid_capstyle="projecting",
            zorder=4,
        )
        ax.scatter(
            [xc],
            [yc],
            s=130,
            facecolors=CURVE_COLOR,
            edgecolors="white",
            linewidths=0.9,
            zorder=6,
            clip_on=False,
        )
        ax.set_xlim(-4.5, 1.4)
        ax.set_ylim(0, 1.15)
        ax.grid(alpha=0.2)
        _strip(ax)
        frames.append(_exp_snap(fig, ax))

    # 3) label for this vertical drop
    for _ in range(EXP_STEP_HOLD):
        fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
        _plot_curve(ax, xs_full, xc)
        for k in range(1, s):
            _stair_h(ax, k)
            _stair_v(ax, k)
        ax.plot(
            [xp, xc],
            [yp, yp],
            color="black",
            linewidth=STAIR_LW,
            solid_capstyle="projecting",
            zorder=4,
        )
        ax.plot(
            [xc, xc],
            [yp, yc],
            color="black",
            linewidth=STAIR_LW,
            solid_capstyle="projecting",
            zorder=4,
        )
        ax.text(
            xc - 0.12,
            0.5 * (yp + yc),
            _v_step_label(s),
            fontsize=ANNOT_SIZE + 2,
            ha="right",
            va="center",
            color="black",
            zorder=5,
        )
        ax.scatter(
            [xc],
            [yc],
            s=130,
            facecolors=CURVE_COLOR,
            edgecolors="white",
            linewidths=0.9,
            zorder=6,
            clip_on=False,
        )
        ax.set_xlim(-4.5, 1.4)
        ax.set_ylim(0, 1.15)
        ax.grid(alpha=0.2)
        _strip(ax)
        frames.append(_exp_snap(fig, ax))

    # 4) move across (next horizontal) + remove label
    if s < 6:
        xp2 = -s
        xc2 = -(s + 1)
        yp2 = float(np.power(2.0, xp2))
        for _ in range(EXP_STEP_HOLD):
            fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
            _plot_curve(ax, xs_full, xp2)
            for k in range(1, s + 1):
                _stair_h(ax, k)
                _stair_v(ax, k)
            ax.plot(
                [xp2, xc2],
                [yp2, yp2],
                color="black",
                linewidth=STAIR_LW,
                solid_capstyle="projecting",
                zorder=4,
            )
            ax.scatter(
                [xc2],
                [yp2],
                s=130,
                facecolors=CURVE_COLOR,
                edgecolors="white",
                linewidths=0.9,
                zorder=6,
                clip_on=False,
            )
            ax.set_xlim(-4.5, 1.4)
            ax.set_ylim(0, 1.15)
            ax.grid(alpha=0.2)
            _strip(ax)
            frames.append(_exp_snap(fig, ax))
    else:
        for _ in range(EXP_STEP_HOLD):
            fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
            _plot_curve(ax, xs_full, xc)
            for k in range(1, s):
                _stair_h(ax, k)
                _stair_v(ax, k)
            ax.plot(
                [xp, xc],
                [yp, yp],
                color="black",
                linewidth=STAIR_LW,
                solid_capstyle="projecting",
                zorder=4,
            )
            ax.plot(
                [xc, xc],
                [yp, yc],
                color="black",
                linewidth=STAIR_LW,
                solid_capstyle="projecting",
                zorder=4,
            )
            ax.scatter(
                [xc],
                [yc],
                s=130,
                facecolors=CURVE_COLOR,
                edgecolors="white",
                linewidths=0.9,
                zorder=6,
                clip_on=False,
            )
            ax.set_xlim(-4.5, 1.4)
            ax.set_ylim(0, 1.15)
            ax.grid(alpha=0.2)
            _strip(ax)
            frames.append(_exp_snap(fig, ax))

save_gif(frames, "38_exponential_mapping.gif", duration=70)

# --- 38: snapshot after one left step with a full drop of 1 to zero ---
fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
_plot_curve(ax, xs_full, 0.0)
_plot_horizontal_only(ax, 1)
ax.plot(
    [-1.0, -1.0],
    [1.0, 0.0],
    color="black",
    linewidth=STAIR_LW,
    solid_capstyle="projecting",
    zorder=4,
)
ax.text(-1.12, 0.5, "1", fontsize=ANNOT_SIZE + 2, ha="right", va="center", color="black", zorder=5)
ax.scatter(
    [-1.0],
    [0.0],
    s=130,
    facecolors=CURVE_COLOR,
    edgecolors="white",
    linewidths=0.9,
    zorder=6,
    clip_on=False,
)
ax.set_xlim(-4.5, 1.4)
ax.set_ylim(0, 1.15)
ax.grid(alpha=0.2)
_strip(ax)
finalize_exponential_figure(fig)
_save_exp_png(fig, "39_step_left_drop_one_to_zero.png")

# --- 39: snapshot at second step with a drop of 1/2 (instead of 1/4) to zero ---
fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
_plot_curve(ax, xs_full, -1.0)
_plot_full_stairs(ax, 1, label_vertical_s=None)
_plot_horizontal_only(ax, 2)
ax.plot(
    [-2.0, -2.0],
    [0.5, 0.0],
    color="black",
    linewidth=STAIR_LW,
    solid_capstyle="projecting",
    zorder=4,
)
ax.text(-2.12, 0.25, "1/2", fontsize=ANNOT_SIZE + 2, ha="right", va="center", color="black", zorder=5)
ax.scatter(
    [-2.0],
    [0.0],
    s=130,
    facecolors=CURVE_COLOR,
    edgecolors="white",
    linewidths=0.9,
    zorder=6,
    clip_on=False,
)
ax.set_xlim(-4.5, 1.4)
ax.set_ylim(0, 1.15)
ax.grid(alpha=0.2)
_strip(ax)
finalize_exponential_figure(fig)
_save_exp_png(fig, "40_step_two_drop_half_to_zero.png")

# --- 41: from end of 38 → remove stairs → solid curve → grow x-range to (-4.5, 6) ---
TRANS_MS = 55
HOLD_MATCH_25 = 5
HOLD_NO_STAIRS = 5
HOLD_SOLID_PREZOOM = 5
N_GROW = 40
xs_wide = np.linspace(-4.5, 6.0, 960)

frames26 = []


def _frame_end_of_25(ax):
    _plot_curve(ax, xs_full, -6.0)
    for k in range(1, 7):
        _stair_h(ax, k)
        _stair_v(ax, k)
    k = 6
    xp = -(k - 1)
    xc = -k
    yp = float(np.power(2.0, xp))
    yc = float(np.power(2.0, xc))
    ax.text(
        xc - 0.12,
        0.5 * (yp + yc),
        _v_step_label(k),
        fontsize=ANNOT_SIZE + 2,
        ha="right",
        va="center",
        color="black",
        zorder=5,
    )
    ax.set_xlim(-4.5, 1.4)
    ax.set_ylim(0, 1.15)
    ax.grid(alpha=0.2)
    _strip(ax)


for _ in range(HOLD_MATCH_25):
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _frame_end_of_25(ax)
    frames26.append(_exp_snap(fig, ax))

for _ in range(HOLD_NO_STAIRS):
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _plot_curve(ax, xs_full, -6.0)
    ax.set_xlim(-4.5, 1.4)
    ax.set_ylim(0, 1.15)
    ax.grid(alpha=0.2)
    _strip(ax)
    frames26.append(_exp_snap(fig, ax))

for _ in range(HOLD_SOLID_PREZOOM):
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    ax.plot(
        xs_wide,
        np.power(2.0, xs_wide),
        color=CURVE_COLOR,
        linestyle="-",
        linewidth=2.0,
        zorder=2,
    )
    ax.set_xlim(-4.5, 1.4)
    ax.set_ylim(0, 1.15)
    ax.grid(alpha=0.2)
    _strip(ax)
    frames26.append(_exp_snap(fig, ax))

for i in range(N_GROW):
    t = i / (N_GROW - 1) if N_GROW > 1 else 1.0
    x0 = -4.5
    x1 = 1.4 + t * (6.0 - 1.4)
    y1 = 2 ** (x1 - 1.4) + 0.15
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    ax.plot(
        xs_wide,
        np.power(2.0, xs_wide),
        color=CURVE_COLOR,
        linestyle="-",
        linewidth=2.0,
        zorder=2,
    )
    ax.set_xlim(x0, x1)
    ax.set_ylim(0.0, y1)
    ax.grid(alpha=0.2)
    _strip(ax)
    frames26.append(_exp_snap(fig, ax))

save_gif(frames26, "41_exponential_transition.gif", duration=TRANS_MS)

# --- 29–30: static (-4.5, 6) reference plots with legends ---
xs_leg = np.linspace(-4.5, 6.0, 700)
fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
ax.plot(
    xs_leg,
    np.power(2.0, xs_leg),
    color=CURVE_COLOR,
    linestyle="-",
    linewidth=2.0,
    label=r"$f(x)=2^x$",
)
ax.set_xlim(-4.5, 6.0)
ax.set_ylim(0.0, 2**(6.0 - 1.4) + 0.15)
ax.grid(alpha=0.2)
ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
ax.legend(loc="upper left", prop={"size": LEGEND_SIZE})
_strip(ax)
finalize_exponential_figure(fig)
_save_exp_png(fig, "42_exponential_2x_legend.png")

fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
ax.plot(
    xs_leg,
    np.exp(xs_leg),
    color=CURVE_COLOR,
    linestyle="-",
    linewidth=2.0,
    label=r"$f(x)=e^x$",
)
ax.set_xlim(-4.5, 6.0)
ax.set_ylim(0.0, float(np.exp(6.0 - 1.4)) + 0.15)
ax.grid(alpha=0.2)
ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
ax.legend(loc="upper left", prop={"size": LEGEND_SIZE})
_strip(ax)
finalize_exponential_figure(fig)
_save_exp_png(fig, "43_exponential_ex_legend.png")

# --- 66: morph (L x, 2^x) vs fixed ((ln2)x, 2^x); same box as 42/43 + conversion in legend ---
from matplotlib.lines import Line2D

_ln2 = float(np.log(2.0))


def _ln66_smoothstep(t):
    t = float(np.clip(t, 0.0, 1.0))
    return t * t * (3.0 - 2.0 * t)


def _draw_ln2_axis_morph(ax, s):
    """s in [0,1]: blue L: 1->ln2; gray is ((ln2)x, 2^x). Same box as 42/43; legend shows conversion only."""
    u = _ln66_smoothstep(s)
    L = 1.0 + u * (_ln2 - 1.0)
    y = np.power(2.0, xs_leg)
    xh_m = xs_leg * L
    xh_t = xs_leg * _ln2
    ax.clear()
    ax.plot(
        xh_t,
        y,
        color="#8c8c8c",
        linestyle=(0, (5, 3)),
        linewidth=1.65,
        zorder=1,
    )
    ax.plot(
        xh_m,
        y,
        color=CURVE_COLOR,
        linestyle="-",
        linewidth=2.0,
        zorder=2,
    )
    ax.set_xlim(-4.5, 6.0)
    ax.set_ylim(0.0, 2 ** (6.0 - 1.4) + 0.15)
    ax.grid(alpha=0.2)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
    h_form = Line2D(
        [],
        [],
        linestyle="none",
        linewidth=0,
        color="none",
        label=r"$2^x=e^{(\ln 2)\,x}$",
    )
    ax.legend(handles=[h_form], loc="upper left", prop={"size": LEGEND_SIZE})
    _strip(ax)


LN66_MS = 52
LN66_HOLD = 10
LN66_MORPH = 52
LN66_HOLD_END = 18
frames_ln66 = []
for _ in range(LN66_HOLD):
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _draw_ln2_axis_morph(ax, 0.0)
    frames_ln66.append(_exp_snap(fig, ax))
for i in range(LN66_MORPH):
    t = i / (LN66_MORPH - 1) if LN66_MORPH > 1 else 1.0
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _draw_ln2_axis_morph(ax, t)
    frames_ln66.append(_exp_snap(fig, ax))
for _ in range(LN66_HOLD_END):
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _draw_ln2_axis_morph(ax, 1.0)
    frames_ln66.append(_exp_snap(fig, ax))
save_gif(frames_ln66, "66_two_pow_axis_ln2_to_exp.gif", duration=LN66_MS)


# --- 44–45: softmax on exp(ST−EL); same duo layout / timing as 32–33 (Scene 6b helpers).
_EXP_YLABEL = r"$e^{\mathrm{ST}-\mathrm{EL}}$"


def _ylim_exp_twin(z_a, z_b):
    m = max(float(np.exp(z_a)), float(np.exp(z_b)))
    return 0.0, max(m * 1.12, 0.5)


def _ylim_exp_stack(z_a, z_b):
    t = float(np.exp(z_a) + np.exp(z_b))
    return 0.0, max(t * 1.08, 0.5)


def _style_twin_exp_axis(ax, ylo, yhi, ylabel):
    ax.set_ylim(ylo, yhi)
    ax.set_ylabel(ylabel, fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.set_xlabel("")
    ax.tick_params(axis="both", labelsize=FONT_SIZE)
    ax.grid(axis="y", alpha=0.22)
    ax.axhline(0.0, color="black", linewidth=1.0, linestyle=":")
    ax.set_xlim(*N_BAR_XLIM)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["A", "B"])


def _draw_twin_exp_bars(ax, z_a, z_b, c_a, c_b, phase, alpha_a=1.0, ylabel=_EXP_YLABEL):
    ax.clear()
    ea, eb = float(np.exp(z_a)), float(np.exp(z_b))
    ylo, yhi = _ylim_exp_twin(z_a, z_b)
    if phase >= 1:
        ax.bar([0], [ea], width=0.52, color=[c_a], edgecolor="black", linewidth=0.9, alpha=alpha_a)
    if phase >= 2:
        ax.bar([1], [eb], width=0.52, color=[c_b], edgecolor="black", linewidth=0.9)
    _style_twin_exp_axis(ax, ylo, yhi, ylabel)


def _e_frac_tex(z):
    zi = int(round(float(z)))
    return rf"e^{{{zi}}}"


def _draw_exp_softmax_stack(ax_r, stage, z_a, z_b, c_a, c_b):
    h1 = float(np.exp(z_a))
    h2 = float(np.exp(z_b))
    ylo, yhi = _ylim_exp_stack(z_a, z_b)
    ax_r.clear()
    ax_r.bar([N_STACK_X], [h1], width=N_STACK_WIDTH, bottom=0.0, color=c_a, edgecolor="black", linewidth=0.9)
    ax_r.bar([N_STACK_X], [h2], width=N_STACK_WIDTH, bottom=h1, color=c_b, edgecolor="black", linewidth=0.9)
    _style_twin_exp_axis(ax_r, ylo, yhi, ylabel=_EXP_YLABEL)
    if stage <= 0:
        return
    ta, tb = _e_frac_tex(z_a), _e_frac_tex(z_b)
    den = f"{ta}+{tb}"
    p_lo = float(h1 / (h1 + h2))
    p_hi = float(h2 / (h1 + h2))
    if stage == 1:
        body = (
            r"$\mathrm{A}:\ \dfrac{" + ta + "}{" + den + r"}$"
            + "\n"
            + r"$\mathrm{B}:\ \dfrac{" + tb + "}{" + den + r"}$"
        )
    elif stage == 2:
        body = (
            rf"$\mathrm{{A}}:\ \approx {p_lo:.3f}$"
            + "\n"
            + rf"$\mathrm{{B}}:\ \approx {p_hi:.3f}$"
        )
    else:
        body = (
            rf"$\mathrm{{A}}:\ {p_lo:.3f}$"
            + "\n"
            + rf"$\mathrm{{B}}:\ {p_hi:.3f}$"
        )
    ax_r.text(
        0.98,
        0.98,
        body,
        transform=ax_r.transAxes,
        ha="right",
        va="top",
        fontsize=NOTE_SIZE + 6,
        linespacing=1.42,
        color="#141414",
        bbox=dict(
            boxstyle="round,pad=0.32",
            facecolor="white",
            edgecolor="#c4c4c4",
            linewidth=0.75,
            alpha=0.96,
        ),
        zorder=NORM_STACK_LABEL_Z,
    )


def _softmax_cadence_core(
    ax0_l,
    ax0_r,
    fig0,
    left_draw,
    z_a,
    z_b,
    c_st_a,
    c_st_b,
    c_exp_a,
    c_exp_b,
    twin_alpha_a,
    twin_z_ylim=None,
):
    out = []
    for _ in range(4):
        left_draw(ax0_l, 0)
        _draw_twin_bars(ax0_r, z_a, z_b, c_st_a, c_st_b, phase=0, z_ylim=twin_z_ylim)
        out.append(_norm_frame_png(fig0))
    for _ in range(5):
        left_draw(ax0_l, 1)
        _draw_twin_bars(ax0_r, z_a, z_b, c_st_a, c_st_b, phase=1, alpha_a=twin_alpha_a, z_ylim=twin_z_ylim)
        out.append(_norm_frame_png(fig0))
    for _ in range(5):
        left_draw(ax0_l, 2)
        _draw_twin_bars(ax0_r, z_a, z_b, c_st_a, c_st_b, phase=2, alpha_a=twin_alpha_a, z_ylim=twin_z_ylim)
        out.append(_norm_frame_png(fig0))
    for _ in range(5):
        left_draw(ax0_l, 2)
        _draw_twin_exp_bars(ax0_r, z_a, z_b, c_exp_a, c_exp_b, phase=1, alpha_a=twin_alpha_a)
        out.append(_norm_frame_png(fig0))
    for _ in range(5):
        left_draw(ax0_l, 2)
        _draw_twin_exp_bars(ax0_r, z_a, z_b, c_exp_a, c_exp_b, phase=2, alpha_a=twin_alpha_a)
        out.append(_norm_frame_png(fig0))
    for st in (0, 1, 2, 3):
        nrep = 6 if st == 3 else 5
        for _ in range(nrep):
            left_draw(ax0_l, 2)
            _draw_exp_softmax_stack(ax0_r, st, z_a, z_b, c_exp_a, c_exp_b)
            out.append(_norm_frame_png(fig0))
    return out


def _frames_exp_softmax_same_cadence(
    ax0_l,
    ax0_r,
    fig0,
    s_a,
    e_a,
    y_a,
    c_a_plane,
    s_b,
    e_b,
    y_b,
    c_b_plane,
    z_a,
    z_b,
    c_st_a,
    c_st_b,
    c_exp_a,
    c_exp_b,
    twin_alpha_a,
):
    def _ld(ax, ph):
        _draw_left_progress(ax, s_a, e_a, y_a, c_a_plane, s_b, e_b, y_b, c_b_plane, phase=ph)

    return _softmax_cadence_core(
        ax0_l,
        ax0_r,
        fig0,
        _ld,
        z_a,
        z_b,
        c_st_a,
        c_st_b,
        c_exp_a,
        c_exp_b,
        twin_alpha_a,
        None,
    )


frames_exp_pos = []
fig_ep, (ax_ep_l, ax_ep_r) = plt.subplots(
    1,
    2,
    figsize=NORM_DUO_FIGSIZE,
    gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28},
)
fig_ep.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
_ce_pos_a, _ce_pos_b = _exp_bar_facecolors([ZA1, ZB4])
frames_exp_pos.extend(
    _frames_exp_softmax_same_cadence(
        ax_ep_l,
        ax_ep_r,
        fig_ep,
        SA1,
        EA1,
        YA1,
        COL_POS_LO,
        SB4,
        EB4,
        YB4,
        COL_POS_HI,
        ZA1,
        ZB4,
        COL_POS_LO,
        COL_POS_HI,
        _ce_pos_a,
        _ce_pos_b,
        1.0,
    )
)
plt.close(fig_ep)
save_gif(frames_exp_pos, "44_norm_exp_positive_softmax.gif", duration=NORM_MS)

frames_exp_neg = []
fig_en, (ax_en_l, ax_en_r) = plt.subplots(
    1,
    2,
    figsize=NORM_DUO_FIGSIZE,
    gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28},
)
fig_en.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
c_na, c_nb = _exp_bar_facecolors([ZAN, ZB4])
frames_exp_neg.extend(
    _frames_exp_softmax_same_cadence(
        ax_en_l,
        ax_en_r,
        fig_en,
        SAN,
        EAN,
        YAN,
        COL_NEG_LO,
        SB4,
        EB4,
        YB4,
        COL_POS_HI,
        ZAN,
        ZB4,
        COL_NEG_LO,
        COL_POS_HI,
        c_na,
        c_nb,
        0.62,
    )
)
plt.close(fig_en)
save_gif(frames_exp_neg, "45_norm_exp_negative_softmax.gif", duration=NORM_MS)

# --- 46–47: far-region zoom-out; duo A/B with ST−EL gaps 101 vs 104; softmax when B on ST−EL=0 and z_A=1 ---
ZOOM_MS = 45
N_ZOOM = 52
ZOOM_X_END = (4.0, 120.0)
ZOOM_Y_END = (4.0, 110.0)
Z_GAP_A, Z_GAP_B = 101.0, 104.0
S_FAR_A, S_FAR_B = 110.0, 115.0
E_FAR_A = S_FAR_A - float(midpoint_shift) - Z_GAP_A
E_FAR_B = S_FAR_B - float(midpoint_shift) - Z_GAP_B
TWIN_Z_LIM_FAR = (0.0, max(Z_GAP_A, Z_GAP_B) * 1.12)


def _threshold_line_xy(ax, xlo, xhi, shift, label="ST - EL = 0"):
    x = np.linspace(xlo, xhi, 600)
    ax.plot(x, x - shift, "--", color="black", linewidth=1.0, label=label)


def _lerp(a0, a1, t):
    return float(a0 + t * (a1 - a0))


def _smoothstep(t):
    t = float(np.clip(t, 0.0, 1.0))
    return t * t * (3.0 - 2.0 * t)


def _frame_zoom_far(ax, x0, x1, y0, y1):
    ax.clear()
    draw_dataset(ax, study_real, exam_real, y_real)
    add_outcome_icon(ax, S_FAR_A, E_FAR_A, passed=True, alpha=0.98, zoom=0.16)
    add_outcome_icon(ax, S_FAR_B, E_FAR_B, passed=True, alpha=0.98, zoom=0.16)
    _threshold_line_xy(ax, x0, x1, float(midpoint_shift), label="ST - EL = 0")
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.grid(alpha=0.2)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
    add_combined_legend(ax, loc="upper left")


def _draw_far_pair_left(ax, phase):
    sh = float(midpoint_shift)
    ax.clear()
    draw_dataset(ax, study_real, exam_real, y_real, alpha=0.22)
    add_outcome_icon(ax, S_FAR_A, E_FAR_A, passed=True, alpha=0.98, zoom=0.16)
    add_outcome_icon(ax, S_FAR_B, E_FAR_B, passed=True, alpha=0.98, zoom=0.16)
    _threshold_line_xy(ax, ZOOM_X_END[0], ZOOM_X_END[1], sh, label="ST - EL = 0")
    if phase >= 1:
        _draw_student_distance(ax, S_FAR_A, E_FAR_A, 1, COL_POS_LO)
    if phase >= 2:
        _draw_student_distance(ax, S_FAR_B, E_FAR_B, 1, COL_POS_HI)
    ax.set_xlim(*ZOOM_X_END)
    ax.set_ylim(*ZOOM_Y_END)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.grid(alpha=0.2)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
    add_combined_legend(ax, loc="upper left")


def _blank_twin_st_axis(ax_r):
    ax_r.clear()
    _style_bar_axis(ax_r, stacked=False, ylabel="ST - EL", z_ylim=TWIN_Z_LIM_FAR)


frames_far = []
Z_FAR_A = float(S_FAR_A - E_FAR_A - float(midpoint_shift))
Z_FAR_B = float(S_FAR_B - E_FAR_B - float(midpoint_shift))
assert np.isclose(Z_FAR_A, Z_GAP_A) and np.isclose(Z_FAR_B, Z_GAP_B)
_ce_fa, _ce_fb = _exp_bar_facecolors([Z_FAR_A, Z_FAR_B])
fig_fz, (ax_fz_l, ax_fz_r) = plt.subplots(
    1,
    2,
    figsize=NORM_DUO_FIGSIZE,
    gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28},
)
fig_fz.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
for i in range(N_ZOOM):
    u = _smoothstep(i / (N_ZOOM - 1)) if N_ZOOM > 1 else 1.0
    xa = _lerp(xlim[0], ZOOM_X_END[0], u)
    xb = _lerp(xlim[1], ZOOM_X_END[1], u)
    ya = _lerp(ylim[0], ZOOM_Y_END[0], u)
    yb = _lerp(ylim[1], ZOOM_Y_END[1], u)
    _frame_zoom_far(ax_fz_l, xa, xb, ya, yb)
    _blank_twin_st_axis(ax_fz_r)
    frames_far.append(_norm_frame_png(fig_fz))
for _ in range(5):
    _draw_far_pair_left(ax_fz_l, 1)
    _blank_twin_st_axis(ax_fz_r)
    frames_far.append(_norm_frame_png(fig_fz))
for _ in range(5):
    _draw_far_pair_left(ax_fz_l, 2)
    _blank_twin_st_axis(ax_fz_r)
    frames_far.append(_norm_frame_png(fig_fz))


def _ld_far_locked(ax, ph):
    _draw_far_pair_left(ax, 2)


frames_far.extend(
    _softmax_cadence_core(
        ax_fz_l,
        ax_fz_r,
        fig_fz,
        _ld_far_locked,
        Z_FAR_A,
        Z_FAR_B,
        COL_POS_LO,
        COL_POS_HI,
        _ce_fa,
        _ce_fb,
        1.0,
        TWIN_Z_LIM_FAR,
    )
)
plt.close(fig_fz)
save_gif(frames_far, "46_far_study_hours_zoom.gif", duration=NORM_MS)

# Same (0,7) scatter as 44/45: A reuses the z=+1 pass pick; B is synthetic on ST−EL=0 (not in the point list).
S_Z1, E_Z1, Y_Z1 = SA1, EA1, YA1
sh_z = float(midpoint_shift)
S_TH, E_TH, Y_TH = 5.0, 5.0 - sh_z, 1
Z_TH_A = float(S_Z1 - E_Z1 - sh_z)
Z_TH_B = float(S_TH - E_TH - sh_z)
assert np.isclose(Z_TH_A, ZA1) and np.isclose(Z_TH_B, 0.0)
_ce_th_a, _ce_th_b = _exp_bar_facecolors([Z_TH_A, Z_TH_B])
frames_th = []
fig_th, (ax_th_l, ax_th_r) = plt.subplots(
    1,
    2,
    figsize=NORM_DUO_FIGSIZE,
    gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28},
)
fig_th.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
frames_th.extend(
    _frames_exp_softmax_same_cadence(
        ax_th_l,
        ax_th_r,
        fig_th,
        S_Z1,
        E_Z1,
        Y_Z1,
        COL_POS_LO,
        S_TH,
        E_TH,
        Y_TH,
        COL_POS_HI,
        Z_TH_A,
        Z_TH_B,
        COL_POS_LO,
        COL_POS_HI,
        _ce_th_a,
        _ce_th_b,
        1.0,
    )
)
plt.close(fig_th)
save_gif(frames_th, "47_softmax_threshold_and_z1.gif", duration=NORM_MS)



def _draw_pf_softmax_stack_snapshot(ax_l, ax_r, caption_body):
    """Same duo as 47 @ phase 2 (left); stacked exp bars; probability labels only in the text box."""
    _draw_left_progress(
        ax_l,
        S_Z1,
        E_Z1,
        Y_Z1,
        COL_POS_LO,
        S_TH,
        E_TH,
        Y_TH,
        COL_POS_HI,
        phase=2,
    )
    z_a, z_b = float(Z_TH_A), float(Z_TH_B)
    h1 = float(np.exp(z_a))
    h2 = float(np.exp(z_b))
    ylo, yhi = _ylim_exp_stack(z_a, z_b)
    ax_r.clear()
    ax_r.bar([N_STACK_X], [h1], width=N_STACK_WIDTH, bottom=0.0, color=_ce_th_a, edgecolor="black", linewidth=0.9)
    ax_r.bar([N_STACK_X], [h2], width=N_STACK_WIDTH, bottom=h1, color=_ce_th_b, edgecolor="black", linewidth=0.9)
    ax_r.set_ylim(ylo, yhi)
    ax_r.set_ylabel(_EXP_YLABEL, fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax_r.set_xlabel("")
    ax_r.tick_params(axis="y", which="major", labelsize=FONT_SIZE)
    ax_r.tick_params(axis="x", which="both", bottom=False, top=False, labelbottom=False)
    ax_r.grid(axis="y", alpha=0.22)
    ax_r.axhline(0.0, color="black", linewidth=1.0, linestyle=":")
    ax_r.set_xlim(*N_BAR_XLIM)
    ax_r.set_xticks([])
    ax_r.text(
        0.98,
        0.98,
        caption_body,
        transform=ax_r.transAxes,
        ha="right",
        va="top",
        fontsize=NOTE_SIZE + 6,
        linespacing=1.42,
        color="#141414",
        bbox=dict(
            boxstyle="round,pad=0.32",
            facecolor="white",
            edgecolor="#c4c4c4",
            linewidth=0.75,
            alpha=0.96,
        ),
        zorder=NORM_STACK_LABEL_Z,
    )

_pf_cap_pass = r"$\mathrm{P}(\mathrm{A\ passes}) = \dfrac{e^{1}}{e^{0}+e^{1}}$"
_pf_snap_specs = (
    (
        "48_softmax_pf_stack_normalized_frac.png",
        _pf_cap_pass + "\n" + r"$\mathrm{P}(\mathrm{A\ fails}) = \dfrac{e^{0}}{e^{0}+e^{1}}$",
    ),
    (
        "49_softmax_pf_stack_fail_sum_ratio.png",
        _pf_cap_pass + "\n" + r"$\mathrm{P}(\mathrm{A\ fails}) = \dfrac{e^{0}+e^{1}-e^{1}}{e^{0}+e^{1}}$",
    ),
    (
        "50_softmax_pf_stack_fail_one_minus.png",
        _pf_cap_pass + "\n" + r"$\mathrm{P}(\mathrm{A\ fails}) = 1 - \mathrm{P}(\mathrm{A\ passes})$",
    ),
    (
        "51_softmax_pf_stack_fail_eneg1.png",
        _pf_cap_pass + "\n" + r"$\mathrm{P}(\mathrm{A\ fails}) = \dfrac{e^{-1}}{e^{-1}+e^{0}}$",
    ),
    (
        "52_softmax_pf_stack_fail_sum_ratio_pass_B.png",
        _pf_cap_pass + "\n" + r"$\mathrm{P}(\mathrm{B}) = \dfrac{e^{0}}{e^{0}+e^{1}}$",
    ),
)

for _pf_fn, _pf_body in _pf_snap_specs:
    _fig_pf, (_ax_pf_l, _ax_pf_r) = plt.subplots(
        1,
        2,
        figsize=NORM_DUO_FIGSIZE,
        gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28},
    )
    _fig_pf.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
    _draw_pf_softmax_stack_snapshot(_ax_pf_l, _ax_pf_r, _pf_body)
    _im_pf = _norm_frame_png(_fig_pf)
    _im_pf.save(OUTPUT_DIR / _pf_fn)
    plt.close(_fig_pf)




## Scene 8 - Sigmoid in 3D (pass/fail mirrored surfaces)

Script alignment:
- map any `ST - EL` value to pass probability with sigmoid
- show mirrored fail probability
- present both in 3D style


In [ ]:
import io
import matplotlib as mpl
from PIL import Image
st_axis = np.linspace(xlim[0], xlim[1], 220)
el_axis = np.linspace(ylim[0], ylim[1], 220)
ST, EL = np.meshgrid(st_axis, el_axis)
DIFF = ST - EL
P_PASS = sigmoid(DIFF)
P_FAIL = sigmoid(-DIFF)
cvals  = [0, .5, 1]
colors = ['red', 'white', 'green']
norm=plt.Normalize(min(cvals),max(cvals))
tuples = list(zip(map(norm,cvals), colors))
CMAP = mpl.colors.LinearSegmentedColormap.from_list("", tuples, 100)

# Threshold curve where ST=EL and sigmoid(0)=1/2
diag_line = np.linspace(max(xlim[0], ylim[0]), min(xlim[1], ylim[1]), 220)
threshold_half = np.full_like(diag_line, 0.5)

# Reference cross-section for readability (fixed exam length)
el_ref = np.full_like(st_axis, np.mean(ylim))

all_study = study_real
all_exam = exam_real
all_diff = all_study - all_exam
pass_idx = y_real == 1
fail_idx = y_real == 0

# Match the red/green palette used in neural-networks.ipynb (21.gif section)
SIG_PASS_COLOR = "green"
SIG_FAIL_COLOR = "red"


def style_sigmoid_axes(ax, az):
    ax.plot(diag_line, diag_line, threshold_half, color="black", linestyle="--", linewidth=1)
    ax.tick_params(axis="x", labelsize=FONT_SIZE)
    ax.tick_params(axis="y", labelsize=FONT_SIZE)
    ax.tick_params(axis="z", labelsize=FONT_SIZE)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_zlabel("Sigmoid Probability", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_zlim(0, 1)
    ax.view_init(elev=26, azim=az)


def scatter_whole_data_by_class(ax, prob_values):
    ax.scatter(all_study[pass_idx], all_exam[pass_idx], prob_values[pass_idx], c=SIG_PASS_COLOR, s=30, alpha=0.95, depthshade=False)
    ax.scatter(all_study[fail_idx], all_exam[fail_idx], prob_values[fail_idx], c=SIG_FAIL_COLOR, s=30, alpha=0.95, depthshade=False)


ROTATION_FRAMES = 120
ROTATION_MS = 32
SIG2D_FIGSIZE = (10.5, 5.4)
SIG2D_DPI = 140
SIG2D_X = np.linspace(-7.0, 7.0, 900)
SIG2D_Y = sigmoid(SIG2D_X)
SIG2D_Y_NEG = sigmoid(-SIG2D_X)
SIG2D_COLOR = "#1f77b4"
SIG2D_MIRROR_PASS = "#2ca02c"
SIG2D_MIRROR_FAIL = "#d62728"
SIG2D_ADJ = dict(left=0.11, right=0.97, top=0.94, bottom=0.12)

from matplotlib import lines as mlines


def _finalize_sig2d(fig):
    fig.subplots_adjust(**SIG2D_ADJ)


def _save_sig2d(fig, filename):
    _finalize_sig2d(fig)
    fig.savefig(
        OUTPUT_DIR / filename,
        format="png",
        dpi=SIG2D_DPI,
        bbox_inches=None,
        pad_inches=SAVE_PAD_INCHES,
    )
    plt.close(fig)


def _sig2d_axes_and_crosshairs(ax):
    ax.set_xlim(-7.0, 7.0)
    ax.set_ylim(0.0, 1.0)
    ax.set_xlabel(r"$z = \mathrm{ST}-\mathrm{EL}$", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.set_ylabel(r"$\sigma(z)$", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.grid(alpha=0.2)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
    ax.axhline(0.5, color="black", linewidth=0.9, linestyle=":", alpha=0.55)
    ax.axvline(0.0, color="black", linewidth=0.9, linestyle=":", alpha=0.55)


def _plot_sigmoid_2d_base(ax):
    _sig2d_axes_and_crosshairs(ax)
    ax.plot(SIG2D_X, SIG2D_Y, color=SIG2D_COLOR, linewidth=2.2)


SIG2D_LEGEND_FS = LEGEND_SIZE + 6
SIG2D_LEGEND_48 = r"$\sigma(z)=\frac{e^{z}}{e^{z}+e^{0}}$"


def _legend_sig2d(ax, label_math):
    h = [mlines.Line2D([], [], linestyle="none", linewidth=0, color="none", label=label_math)]
    ax.legend(
        handles=h,
        loc="upper left",
        prop={"size": SIG2D_LEGEND_FS},
        frameon=True,
        borderaxespad=0.9,
        labelspacing=0.9,
    )


def _sig2d_frame_png(fig):
    _finalize_sig2d(fig)
    buf = io.BytesIO()
    fig.savefig(
        buf,
        format="png",
        dpi=SIG2D_DPI,
        bbox_inches=None,
        pad_inches=SAVE_PAD_INCHES,
    )
    buf.seek(0)
    im = Image.open(buf).convert("RGB").copy()
    buf.close()
    return im


def _sig2d_smoothstep(t):
    t = float(np.clip(t, 0.0, 1.0))
    return t * t * (3.0 - 2.0 * t)


def _sig2d_draw_reveal(ax, x_right):
    """Axes, crosshairs, legend (same as 48), and sigmoid curve for z <= x_right."""
    ax.clear()
    _sig2d_axes_and_crosshairs(ax)
    _legend_sig2d(ax, SIG2D_LEGEND_48)
    m = SIG2D_X <= x_right
    if np.count_nonzero(m) >= 2:
        ax.plot(
            SIG2D_X[m],
            SIG2D_Y[m],
            color=SIG2D_COLOR,
            linewidth=2.2,
            solid_capstyle="round",
        )


SIG2D_REVEAL_MS = 40
SIG2D_REVEAL_HOLD = 16
SIG2D_REVEAL_ANIM = 88
SIG2D_REVEAL_TAIL = 18

frames_s2d_reveal = []
fig_s2d_r, ax_s2d_r = plt.subplots(figsize=SIG2D_FIGSIZE)
for _ in range(SIG2D_REVEAL_HOLD):
    ax_s2d_r.clear()
    _sig2d_axes_and_crosshairs(ax_s2d_r)
    _legend_sig2d(ax_s2d_r, SIG2D_LEGEND_48)
    frames_s2d_reveal.append(_sig2d_frame_png(fig_s2d_r))
for i in range(SIG2D_REVEAL_ANIM):
    u = _sig2d_smoothstep(i / (SIG2D_REVEAL_ANIM - 1)) if SIG2D_REVEAL_ANIM > 1 else 1.0
    xr = -7.0 + 14.0 * u
    _sig2d_draw_reveal(ax_s2d_r, xr)
    frames_s2d_reveal.append(_sig2d_frame_png(fig_s2d_r))
for _ in range(SIG2D_REVEAL_TAIL):
    _sig2d_draw_reveal(ax_s2d_r, 7.0)
    frames_s2d_reveal.append(_sig2d_frame_png(fig_s2d_r))
plt.close(fig_s2d_r)
save_gif(frames_s2d_reveal, "53_sigmoid_2d_normalized_exp_reveal.gif", duration=SIG2D_REVEAL_MS)


def _sig2d_draw_mirror_reveal_frame(ax, x_fail_right):
    """Green sigma(z) always; red sigma(-z) revealed for z <= x_fail_right."""
    ax.clear()
    _sig2d_axes_and_crosshairs(ax)
    ax.plot(
        SIG2D_X,
        SIG2D_Y,
        color=SIG2D_MIRROR_PASS,
        linewidth=2.35,
        solid_capstyle="round",
        zorder=3,
    )
    m = SIG2D_X <= x_fail_right
    if np.count_nonzero(m) >= 2:
        ax.plot(
            SIG2D_X[m],
            SIG2D_Y_NEG[m],
            color=SIG2D_MIRROR_FAIL,
            linewidth=2.35,
            solid_capstyle="round",
            zorder=4,
        )
    h_p = mlines.Line2D([], [], color=SIG2D_MIRROR_PASS, linewidth=2.6, label=r"$\sigma(z)$")
    h_f = mlines.Line2D([], [], color=SIG2D_MIRROR_FAIL, linewidth=2.6, label=r"$\sigma(-z)$")
    ax.legend(handles=[h_p, h_f], loc="upper right", fontsize=LEGEND_SIZE, frameon=True, borderaxespad=0.55)


SIG2D_MIRROR_REVEAL_MS = SIG2D_REVEAL_MS
frames_s2d_mirror = []
fig_s2d_m, ax_s2d_m = plt.subplots(figsize=SIG2D_FIGSIZE)
for _ in range(SIG2D_REVEAL_HOLD):
    _sig2d_draw_mirror_reveal_frame(ax_s2d_m, -7.0)
    frames_s2d_mirror.append(_sig2d_frame_png(fig_s2d_m))
for i in range(SIG2D_REVEAL_ANIM):
    u = _sig2d_smoothstep(i / (SIG2D_REVEAL_ANIM - 1)) if SIG2D_REVEAL_ANIM > 1 else 1.0
    xr = -7.0 + 14.0 * u
    _sig2d_draw_mirror_reveal_frame(ax_s2d_m, xr)
    frames_s2d_mirror.append(_sig2d_frame_png(fig_s2d_m))
for _ in range(SIG2D_REVEAL_TAIL):
    _sig2d_draw_mirror_reveal_frame(ax_s2d_m, 7.0)
    frames_s2d_mirror.append(_sig2d_frame_png(fig_s2d_m))
plt.close(fig_s2d_m)
save_gif(frames_s2d_mirror, "54_sigmoid_sigma_negz_reveal.gif", duration=SIG2D_MIRROR_REVEAL_MS)


fig_s2a, ax_s2a = plt.subplots(figsize=SIG2D_FIGSIZE)
_plot_sigmoid_2d_base(ax_s2a)
_legend_sig2d(ax_s2a, SIG2D_LEGEND_48)

_save_sig2d(fig_s2a, "55_sigmoid_2d_normalized_exp_legend.png")

fig_s2b, ax_s2b = plt.subplots(figsize=SIG2D_FIGSIZE)
_plot_sigmoid_2d_base(ax_s2b)
_legend_sig2d(
    ax_s2b,
    r"$\sigma(z)=\frac{e^{z}\cdot e^{-z}}{\left(e^{z}+e^{0}\right)\cdot e^{-z}}$",
)
_save_sig2d(fig_s2b, "56_sigmoid_2d_multiply_exp_legend.png")

fig_s2c, ax_s2c = plt.subplots(figsize=SIG2D_FIGSIZE)
_plot_sigmoid_2d_base(ax_s2c)
_legend_sig2d(
    ax_s2c,
    r"$\sigma(z)=\frac{1}{1+e^{-z}}$",
)
_save_sig2d(fig_s2c, "57_sigmoid_2d_standard_legend.png")


rotation_azimuths = np.linspace(25, 345, ROTATION_FRAMES)


# Animated 3D rotation around pass/fail sigmoid surfaces (correct ST/EL axes)
frames = []
for az in rotation_azimuths:
    fig = plt.figure(figsize=(10.5, 5.4))
    ax = fig.add_subplot(111, projection="3d")

    ax.plot_surface(ST, EL, P_PASS, alpha=0.30, color=SIG_PASS_COLOR, linewidth=0, antialiased=True)
    ax.plot_surface(ST, EL, P_FAIL, alpha=0.24, color=SIG_FAIL_COLOR, linewidth=0, antialiased=True)

    style_sigmoid_axes(ax, az)
    ax.text2D(0.02, 0.95, r"$\sigma(z)=\frac{1}{1+e^{-z}}$", transform=ax.transAxes, fontsize=TITLE_SIZE)
    frames.append(fig_to_image(fig, dpi=150))

save_gif(frames, "58_sigmoid_and_mirror.gif", duration=ROTATION_MS)

# Static 3D snapshot
fig = plt.figure(figsize=(10.5, 5.4))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(ST, EL, P_PASS, alpha=0.30, color=SIG_PASS_COLOR, linewidth=0, antialiased=True)
ax.plot_surface(ST, EL, P_FAIL, alpha=0.24, color=SIG_FAIL_COLOR, linewidth=0, antialiased=True)
style_sigmoid_axes(ax, 58)
ax.text2D(0.02, 0.95, r"$\sigma(z)=\frac{1}{1+e^{-z}}$", transform=ax.transAxes, fontsize=TITLE_SIZE)
save_fig(fig, "59_sigmoid_static.png")

# Additional requested 3D sigmoid plots (no formula text)
from matplotlib.colors import LinearSegmentedColormap, to_rgb

extra_angles = rotation_azimuths

# 1) Green-only pass sigmoid with whole data (class-colored)
frames = []
for az in extra_angles:
    fig = plt.figure(figsize=(10.5, 5.4))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, P_PASS, alpha=0.26, color=SIG_PASS_COLOR, linewidth=0, antialiased=True)
    scatter_whole_data_by_class(ax, sigmoid(all_diff))
    style_sigmoid_axes(ax, az)
    frames.append(fig_to_image(fig, dpi=150))
save_gif(frames, "60_sigmoid_pass_green.gif", duration=ROTATION_MS)

# 2) Fail σ(−z): freeze last view of 60, morph surface + point heights σ(z)→σ(−z) and green→red, then orbit
_pass_rgb = np.array(to_rgb(SIG_PASS_COLOR))
_fail_rgb = np.array(to_rgb(SIG_FAIL_COLOR))
_last_az_pass = float(extra_angles[-1])
_z_pass_pts = sigmoid(all_diff)
_z_fail_pts = sigmoid(-all_diff)
_SIG61_HOLD_LAST = 14
_SIG61_MORPH = 84

frames = []
for _ in range(_SIG61_HOLD_LAST):
    fig = plt.figure(figsize=(10.5, 5.4))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, P_PASS, alpha=0.26, color=SIG_PASS_COLOR, linewidth=0, antialiased=True)
    scatter_whole_data_by_class(ax, _z_pass_pts)
    style_sigmoid_axes(ax, _last_az_pass)
    frames.append(fig_to_image(fig, dpi=150))
for i in range(_SIG61_MORPH):
    u = i / (_SIG61_MORPH - 1) if _SIG61_MORPH > 1 else 1.0
    su = _sig2d_smoothstep(u)
    Z_surf = (1.0 - su) * P_PASS + su * P_FAIL
    c_surf = tuple((1.0 - su) * _pass_rgb + su * _fail_rgb)
    z_pts = (1.0 - su) * _z_pass_pts + su * _z_fail_pts
    fig = plt.figure(figsize=(10.5, 5.4))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, Z_surf, alpha=0.26, color=c_surf, linewidth=0, antialiased=True)
    scatter_whole_data_by_class(ax, z_pts)
    style_sigmoid_axes(ax, _last_az_pass)
    frames.append(fig_to_image(fig, dpi=150))
for az in extra_angles:
    fig = plt.figure(figsize=(10.5, 5.4))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, P_FAIL, alpha=0.26, color=SIG_FAIL_COLOR, linewidth=0, antialiased=True)
    scatter_whole_data_by_class(ax, _z_fail_pts)
    style_sigmoid_axes(ax, az)
    frames.append(fig_to_image(fig, dpi=150))
save_gif(frames, "61_sigmoid_fail_red.gif", duration=ROTATION_MS)


frames = []
for az in extra_angles:
    fig = plt.figure(figsize=(10.5, 5.4))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, sigmoid(ST - EL), alpha=0.32, cmap=CMAP, linewidth=0, antialiased=True, shade=False)
    scatter_whole_data_by_class(ax, sigmoid(all_diff))
    style_sigmoid_axes(ax, az)
    frames.append(fig_to_image(fig, dpi=150))
save_gif(frames, "62_sigmoid_pass_colormap.gif", duration=ROTATION_MS)

# 4) Tilt up to top-down so it connects to prior 2D dataset views,
# then rotate counterclockwise by 32 degrees at the top.
TOPDOWN_FRAMES = 90
TOPDOWN_TURN_FRAMES = 40
TOPDOWN_END_HOLD_FRAMES = 12  # duplicate final view after rotation
topdown_elevations = np.linspace(26, 89, TOPDOWN_FRAMES)
topdown_azimuth = 180 + 58
counterclockwise_turn = np.linspace(topdown_azimuth, topdown_azimuth + 32, TOPDOWN_TURN_FRAMES)
frames = []

# Phase 1: tilt upward
for elev in topdown_elevations:
    fig = plt.figure(figsize=(10.5, 5.4))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, sigmoid(ST - EL), alpha=0.32, cmap=CMAP, linewidth=0, antialiased=True, shade=False)
    scatter_whole_data_by_class(ax, sigmoid(all_diff))
    ax.plot(diag_line, diag_line, threshold_half, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_zlabel("Sigmoid Probability", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.zaxis.label.set_visible(elev < topdown_elevations[-1])
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_zlim(0, 1)
    ax.view_init(elev=elev, azim=topdown_azimuth)
    frames.append(fig_to_image(fig, dpi=150))

# Phase 2: at top, rotate counterclockwise 32 degrees
for az in counterclockwise_turn:
    fig = plt.figure(figsize=(10.5, 5.4))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, sigmoid(ST - EL), alpha=0.32, cmap=CMAP, linewidth=0, antialiased=True, shade=False)
    scatter_whole_data_by_class(ax, sigmoid(all_diff))
    ax.plot(diag_line, diag_line, threshold_half, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_zlabel("Sigmoid Probability", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.zaxis.label.set_visible(False)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_zlim(0, 1)
    ax.view_init(elev=89, azim=az)
    frames.append(fig_to_image(fig, dpi=150))

hold = frames[-1]
for _ in range(TOPDOWN_END_HOLD_FRAMES):
    frames.append(hold.copy())

save_gif(frames, "63_sigmoid_colormap_to_topdown.gif", duration=ROTATION_MS)

# 5) Top-down 2D colormap counterpart (σ(0.5·ST − 0.5·EL); same CMAP / alpha / levels as contour reveal)
fig, ax = plt.subplots(figsize=(8, 6))
Z_sig = sigmoid(0.5 * ST - 0.5 * EL)
ax.contourf(
    ST,
    EL,
    Z_sig,
    levels=np.linspace(0.0, 1.0, 45),
    cmap=CMAP,
    vmin=0,
    vmax=1,
    antialiased=True,
    alpha=0.32,
)
add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)

for s, e, lbl in zip(all_study, all_exam, y_real):
    add_outcome_icon(ax, s, e, passed=bool(lbl), zoom=0.16, alpha=0.95)

ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=8)
ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=8)
ax.grid(alpha=0.12)
ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "64_sigmoid_colormap_topdown_2d.png")


## Scene 8c — Jar riddle (bacteria doubling)

Matches `script.md`: full on day 1000, **half** the previous amount each day going backward (`65_jar_riddle_halving.gif`).


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

# Bacteria-in-a-jar riddle: doubles every day; full after 1000 days → half as full each day backward.
JAR_DAYS = 1000
JAR_MS = 12  # ms per frame (~12 s total)
FIG_JAR = (5.2, 6.0)
DPI_JAR = 92
JAR_W = 0.82
JAR_X0 = -0.5 * JAR_W

frames_jar = []
for day in range(JAR_DAYS, -1, -1):
    n_half = JAR_DAYS - day
    h = float(1000.0 * (0.5**n_half))
    fig, ax = plt.subplots(figsize=FIG_JAR)
    ax.set_xlim(JAR_X0 - 0.14, JAR_X0 + JAR_W + 0.14)
    ax.set_ylim(0, 1000)
    ax.axhspan(0, h, facecolor="#4f8fc9", edgecolor="none", zorder=1)
    ax.axhspan(h, 1000, facecolor="#e6e6e6", edgecolor="none", zorder=0)
    ax.add_patch(
        Rectangle(
            (JAR_X0, 0),
            JAR_W,
            1000,
            fill=False,
            edgecolor="black",
            linewidth=1.75,
            zorder=2,
        )
    )
    ax.set_xticks([])
    ax.tick_params(axis="x", bottom=False)
    ax.set_yticks([1000])
    ax.tick_params(axis="y", labelsize=FONT_SIZE)
    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.grid(False)
    leg_h = [
        Line2D(
            [],
            [],
            linestyle="none",
            marker="",
            markersize=0,
            label=f"Day {day}",
        ),
    ]
    ax.legend(handles=leg_h, loc="upper right", frameon=True)
    fig.subplots_adjust(left=0.12, right=0.96, top=0.96, bottom=0.06)
    frames_jar.append(fig_to_image(fig, dpi=DPI_JAR, tight_layout=False))

save_gif(frames_jar, "65_jar_riddle_halving.gif", duration=JAR_MS)


## Scene 9 - Run all exports in order

Execute all cells top-to-bottom once. The `renders/` folder will contain:

Intro (`00`–`16`): `00`/`01` hold empty axes while emphasizing **Study time** then **Exam length** on the axis labels; `02_axes_only.png` is both labels regular weight; `03_dataset_build.gif` holds no points briefly, then adds one point at a time in order `(3,2,1)` then `(2,3,0)` then shuffled rest, with many more frames on the first two points; fail point crosses unseen boundary; threshold shading/labels progression.

After **`24_parallel_lines_additive.gif`**, **`25_fail_point_crosses_threshold_st_el_0.gif`** repeats the **`04`** motion (fail example slides right at fixed exam length) with **ST - EL = 0** drawn.

Scene 6b (`31`–`36`): plane + vertical **ST − EL** bars (greens / red); **`32`** positive stack with **$1/(1{+}4)\!\to\!1/5\!\to\!20\%$**; **`33`**/**`34`** GIFs step through **$(-1,4)$** and **$|z|$** proportion fallacies; **`35`/`36`** bar walkthroughs on **$z\in[-3,3]$** (**$|z|$** vs **$2^{z}$**); same **red–white–green** colormap as sigmoid **3D**.

Scene 5 (`26`–`29`): `26_not_realistic_transition.gif` holds the **separable** students plus threshold, then adds only the **noisy** students one at a time (**(4,3)** first, then **(3,4)**, then the rest), with axis ticks/labels and legend like the other dataset views. **`27_sigmoid_colormap_topdown_reveal.gif`** animates the top-down **σ(0.5·ST − 0.5·EL)** field (half-plane reveal). **`28_parallel_lines_additive_realistic.gif`** replays the **`24`**-style additive parallel lines on the **realistic** plane (**`ST - EL = 0`** labeled only; lines through **3**). `29_proportions_60_80_100.gif` begins and stays on the **full** realistic dataset with the **black** threshold, then layers the colored parallel lines at `ST - EL = 1, 2, 3` and pass-rate text on top (same counting cadence as before).

1. `00_axes_study_time_bold.gif`
2. `01_axes_exam_length_bold.gif`
3. `02_axes_only.png`
4. `03_dataset_build.gif`
5. `04_fail_point_crosses_threshold.gif`
6. `05_threshold_unlabeled.png`
7. `06_threshold_green_right.png`
8. `07_threshold_red_left.png`
9. `08_threshold_micro_drift.gif`
10. `09_dataset_extra_pass_shifted_threshold.png`
11. `10_dataset_two_extra_no_threshold.png`
12. `11_threshold_labeled_st_eq_el.png`
13. `12_threshold_st_eq_el_dot_33.png`
14. `13_threshold_green_labeled_st_gt_el.png`
15. `14_threshold_red_labeled_st_lt_el.png`
16. `15_threshold_label_st_minus_el_eq_el_minus_el.png`
17. `16_threshold_label_st_minus_el_eq_0.png`
18. `17_dataset_overview.png`
19. `18_dataset_overview_labeled.png`
20. `19_dataset_points_on_threshold.gif`
21. `20_threshold_green_st_minus_el_gt_0.png`
22. `21_threshold_red_st_minus_el_lt_0.png`
23. `22_dataset_st2_el1_marker_ticks.gif`
24. `23_parallel_lines_step_01.png` … `23_parallel_lines_step_05.png`
25. `24_parallel_lines_additive.gif`
26. `25_fail_point_crosses_threshold_st_el_0.gif`
27. `26_not_realistic_transition.gif`
28. `27_sigmoid_colormap_topdown_reveal.gif`
29. `28_parallel_lines_additive_realistic.gif`
30. `29_proportions_60_80_100.gif`
31. `30_between_60_and_80.png`
32. `31_norm_positive_two_bars.png`
33. `32_norm_positive_stack.gif`
34. `33_norm_negative_invalid.gif`
35. `34_norm_abs_pitfall.gif`
36. `35_norm_abs_bar_hint.gif`
37. `36_norm_exp_bar_hint.gif`
38. `37_exponential_point_guides.gif`
39. `38_exponential_mapping.gif`
40. `39_step_left_drop_one_to_zero.png`
41. `40_step_two_drop_half_to_zero.png`
42. `41_exponential_transition.gif`
43. `42_exponential_2x_legend.png`
44. `43_exponential_ex_legend.png`
45. `66_two_pow_axis_ln2_to_exp.gif`
46. `44_norm_exp_positive_softmax.gif`
47. `45_norm_exp_negative_softmax.gif`
48. `46_far_study_hours_zoom.gif`
49. `47_softmax_threshold_and_z1.gif`
50. `48_softmax_pf_stack_normalized_frac.png`
51. `49_softmax_pf_stack_fail_sum_ratio.png`
52. `50_softmax_pf_stack_fail_one_minus.png`
53. `51_softmax_pf_stack_fail_eneg1.png`
54. `52_softmax_pf_stack_fail_sum_ratio_pass_B.png`
55. `53_sigmoid_2d_normalized_exp_reveal.gif`
56. `54_sigmoid_sigma_negz_reveal.gif`
57. `55_sigmoid_2d_normalized_exp_legend.png`
58. `56_sigmoid_2d_multiply_exp_legend.png`
59. `57_sigmoid_2d_standard_legend.png`
60. `58_sigmoid_and_mirror.gif`
61. `59_sigmoid_static.png`
62. `60_sigmoid_pass_green.gif`
63. `61_sigmoid_fail_red.gif`
64. `62_sigmoid_pass_colormap.gif`
65. `63_sigmoid_colormap_to_topdown.gif`
66. `64_sigmoid_colormap_topdown_2d.png`
67. `65_jar_riddle_halving.gif`
